# T024 - request context in the selected neural models

Same T014 recipes and protocol (original primary holdout, 30-epoch cap, 300-step validation, patience 5), with label-free within-request features appended. No test predictions or competition submission.

In [ ]:
import sys
from pathlib import Path

import base64
import hashlib
from importlib.metadata import version
import subprocess
import tempfile

WHEELS = [
  {
    "filename": "rtdl_num_embeddings-0.0.12-py3-none-any.whl",
    "sha256": "87fd61270118915cf40888f2164f3a1f0353edf702f15fd91e71a147d23c930d",
    "base64": "UEsDBBQAAAAIABNyjVqA2k94Lh4AAPx0AAAWAAAAcnRkbF9udW1fZW1iZWRkaW5ncy5wee09a3PbOJLf/StwTl3psRRtOZlMVjdKnZM4O6lKJqnYufOd1yVRImRzTZEKScXSpuZ++3U33iQo28nu7NxDNTWR8Gg0Go1+oQHv7++/z9jJcsbjOMmuSrbIC/bLesmLZB6l7DWPqnXBS5Zk7CyardOoYK84X7G3PCoy6BDu7+/v7U0mX3hRJnk2mbAx6xyGh+HwqIPlUZpS2cUeg0/nbZJBRzNcJ7DLP/K3n5p1HwCXPE7mnpqEz/ltUvI2sPUG2TzHelU9z5erdcUnsyTDLpd7e8lylRcVW0bVtfp+KyZa7i2KfMmq7Qp+MFl3nG0D9japeBGlAXu/qoAE+O1TBl/29qpiO6KRZPPyJkWyhVXBOYtK9XuCv/f4Zs5XFXtDTU+KIi9EX7sREPKXPOMazyov5tfOjzDLEHKWSXSxTA1/xrMyL6wKaByuoiJacpiBavVBFVj4iy6f46VqhN9bMaaGCtO9mC/YZH7N5zeTJANyT8rraMW7m5HEJ2B8s+LziseTbLKQ/DYChqt6bPCcoEgiLtgmzOJkyX5iQ1GEnyKC9WX/FqVrTjh0dQ1h3jm75owGZst1WbHr6AvQvmJAVPgFwBlABERgwQJ2nd9y4OQR+ypGGv/a0dB6BgmawcVgeMn+aezF/p7IEW4p4qFxYPmCVRrjW1hKBR+WjM14pzY79tUz/q96JgHORKPrTmdvbw5jl6y+ebrAFe/yeJ3ynpgIbHHRhnFXTszzrEqydb6Gn3LocI+69PunOGi/L34O2Bucz4hNu/2AGUR7U1n9fl356gMWT/SY0FjCPtlEy1XKSwX++fPnbBZV8+tJmfwVN8mRLs4miKQGCHWPdd0Gfol9UERZnHUNiKDer6c7WQhB9ye6HDm+SUkXijsd0zW84tUkJxLI3SEqBXKngFD34nHAnlxafbqbnljYRsujgFmNST7jF9qHsAWTajLpljxd2ISm/eag59uBCp76flxcWbyOHxsisnG2BngFMrWXV+yuztjY19AZlyT0YgDb0QzJfhqzQxefxv5bdKz2JBBmnK3yMqmSL9wWAKYZ7JqePaDNAfca0e6wa0irHY2p4ZbrFS+6vVAvn8EHFzK85cnVdQUMqGV3VzAEX66qbbdtP9WgzJKo/F4YUA+srHVK2e0Z5mvUYQ8Ph8WT4nNZ4HSsyUkhBiK332eDw/AH3RykFVIlXGcJyKSl4G3ZK2ADCS1QYHt3dEQqeLvpeTQ2q56H2YXOfvkTr4ihRSdGndhtUl1DAVWQ4DE6oAxtFi84kD1r0sIgBNiDkRLLTa0VK6EkvjrovMppUNmLrUAHOAP6lHXgWYvDy14dSUGAKI7ny3Vq09NZk81FGIYBrfplXQ25JqBPFZ0mKPtZlmeD9P+1kujUop0emxGNeqpT+G+qoh4fPURH6dbfpaRoor97TXWXDJe87DUidiyO7h/NQZ1E6IAADNg2uMzd34fUElP7bYTXRqkNMShwYLPKkMqulhhvtECaKMfTJ4Xo31/en50Awj+/OWXv3r/69PaEnf78/tPbV1jBXpywT6cnr9irNx9PXp69/Q8pg874/DpD3zoF17G6TsC1LlnE6qJMU5q08iJPUzATYjbbiq0EJFhJ7JiZThky9ChKgM6NIwF2RpQmf6UW1LsLdIfhEBo1mZbJ1TKasqi4Ascfrb5VGm0Bq0y6exFssiJPJWc/aLPeyH9piBFbpHnksyrBrKIWZFCFd5tUovEuY4paPMyMmgioY4HKN5tYN99vFDks/xGbiCXXzRyef0SMePHqzevXl1bhz7zg6FxXBV9y9gXJV7Lufx2Gj/+ZrYp8Fs2SNKm2PVh3jl5mmsyTKt2y6EuexMAdf0HqJhmbR6Ul0x6B0pTIrNC/hK/QHR3VLEewFTijKIQoZOAOboDM8nUWq+0oyd6XOhA/yjSrinU2n2RgnUVpzbADLgns/gGLxgOCG7DZmL70flcmEoqfI5glxpbCVQLfbMbqO1aR00nw2Tyquhfya16C4ApkRZmgGLsEm3XoFWaP2OQX6b6TpCnRfkoWCazwPF9tUU7EPF1jLEi2G0Gf66palaODg+26iH44Cq9AGq1nYZIfYNuDsopmKT+IVsnBFc94EVU8PqhBCa+rZapEqSzzSdJfWL/kyNkV7ysxCOIHmFzEIpUStsw8KUqBEXNkP3abF3EpYno8Ap3U6IEzr8CcKtHQF4IvqUqW32Ywd5ChiL8z9i5J52xuEHlSwiUN2Qfiu16E0nwE7J+nsLBnxVpYRh6B+L0un4OPi8rD/L56XxTV1IOnIJQpxvebCrsYCDIpys8tDuLg6Bs8RAUxsKA7Lr+hU4JyrqotVvsg2pv0D9Eun8QaWFLKLniQrDJh038CQ/kOzerU4qejhQdwJNoCsGsyUBIk+kqyUkDKgwkCZRhMrRmFQacJkZU5m26mdjhWWpdpcsNZ98Xx2cufJ6dv/vMEpOHk9cnx2aePJ6cBezU5effi5NWrN7/8qRe6cM1iwew5sIYOetprLagwYMMeG12ysY99RoPh5Z4jfTchCQ4wMXgXNM7QFegb9q82lHv2vB9HCfh/ME3bTdXmEYlP0p7c6SfD8pUgCUF9e61LQZlTztnHk+NX704ITMyrKEn/r7rYR24E2LcQOxzsAFi+4uPXEcjShzvbRw8KCNut/wY4o/L6h6D8iH1469gC3zGX8bPAYnCxDt+3Js8eMr9nJvyBhnWWfwYX6eSHw+GdVofRbTULw9KTjSjJ0RNT3XdAFPzzmmfzRELBg41npoGq3gpcyL2Urhwjd21o2hpqOmaOaYDEFVVBm+3zrYGb74zWeGhRg2+qSPLZtqYLBj/gZYNcB2kKMn7/Zh89qVPpGz0OH+NP7UX13N5+ejedeenoIypkASQFKFTbit2NE7lND8GL+KZ/pmIWdnTgegvNtcXW76Mz5NcSNjCbWUAnTmnPTcmnZBi/shooNRmtwHnhcROxdzIWMEV2mxK4qRZU06nbQTAhtCHeVGEuWGWbejgk7N4C/QVe3XKesShNW7gMP22KUdU/JByo9e/YjkNZUszh1MDLM94I40gkJ1yAfSAMu0B7Z8btBMoQiZwpesMMooLOsaEDOPpFItYLIwIgSgqwQeZAQWCrCMzCxYIXHPiF2KsGpBvNVGjxLJp9ZMscPMoaC+J5H3KBYZzGKjRs2o5mA2JbGUkjK/b2GhfVKABsZB321UgnAqvSk0VXvrYG3uAsukmjHRCNa2wt7n2B74j8IqmsGuOtadXyG0eEM9sWc3i8HjPx6TDo022jYmt/yTFJmWQliKm5mKLsFZjFdFecSNUKHh0E1bLl/ItU/I4T079XVGpTJ6w3+t0MjCt3pClt/U7J/YLoVt4P5ld18X8jkBJldSHmdtnM78ExqZ0B2NjNQtZg7s6C2go/knqibl9VW7mDUQYngZLXhASKIZ5hdltUccKo58SgEYrFLXbXQC3Ogz1og+SMU6Rf4P5he0bJWRWBLUPWdJUXnZ9VGg9O5msy/nXEvlagawn3i+SyZ6f0uCyZLJyp6xDA8METOEE7B2ToUp0pADwxDePC2zEALQBC35SYb05yOo2kq8aUUp45y9JjP7GjB8/o2NLjvrSw6janSfL4ChT9vWeByN1rYZDPnMVJygXqbXAqQlBQ3Ycz2QuFrdkO6PZEUJxF2UGSLQ4G8L875qLOo7B7hQpT2ZZfk18HlTZ2d8zMWRwKp7DnY2euF8PRJcwy237/LGE/wdYBe/bOabXNJND8LMDiUQhOXoB9EB+Ox3VGVJmjIX5pzkakJrr4AB+W4uDFYcEALJUENuGSR5kMwFGwLfFtsQ47rYqEdmO54tEN+ezrEpUoRv+zq1TMWFLEB2Cl8mZ1ioXMnAWln6/TWIjbAoCDG7AFroEhfHBmYBgkFUXf8VwSvAP0PAQuKhl2VfBVkc95iaXhuyR7F21O0Xwt6uQXOsVO2hVEPXfDpcKvzCZC4bjurHR7MZd2cgPLAs6lTtu9AI1ZXZQVMMxxtr28lCmsosfWaifVl1Nf8KsCp4CujG6IXq7bDNhxlpdc+8Yi4rBHmtBWjTpm91LMVrMpHWyBkS1d0GlLhvMU1iVu1uqoyFSH6z6V0ZUO16nRSGWJZfq8Bv8uSfHo0LiJR+HQdRRHJh53PqkKFDtuiG14CB8wbE10hMYYu8sp+/ZasYn5PKE0WVzBGkpH343S1tsOt7pCbDf6VBmGIfGr7BI4hdvx1ldsuGdsoiWq0mLW8dfOMskmpYiCTmADLToj9vRJwKgc/PF1kVTbCZAJNmfJoXLIB09+NQAlad2wyrmILhBiSOWmi6t2kxsQIdWl2zib6oZv8VRO5xRIdrUz4MNXcjHP4MdHQYG8mDqbvkt+fI0802mP7QT2EoPieMR5FzQRcJjWnEyR1oHOV4L2wXRazpObpBrQaM/Hw/AweH40nYqEbEbWYpraMYltjZ5pNAOngnV1vkKRf6EjdsIH0Qdolslt4WPLFfBW6bizoqxxAon6yjSRpe6cv2FQLaVkhEQIk6m1wG5X1KJRhi0+x8vp1D++plMPfEawv0Ds002Ca0KNpoBaDgNaSVUh2WiPS0H1kRwLi2ePbSdAam9kMSdEw15DCWpKWTIyZzsDGnkZbZLleokZJGUyS2vMLcEmyANiB7A/sKEdTJJQYJnvB0V3boZ8sbTpgpx7/A5PPsy55WC4joWdE0Oew3lPJzTDcOfaLzi6zwCOrY8minH27YHOlSFvjaO8c9dgv2MoxyIv8tvaKAqmb6Th5R33RFpGQnaZ5+l6mfnGGjpj4WKd7zDem97r+V32uYbdPUeT8hxzSAgsEvSwYTjfccEEHR0xmRK5sZW0MeylJJtXKlPHNb0sU/pcoV1quHQGTDarcP5sSKHn/ozcST+BK4piXP4EL4EU7f3nJjsqxl/mBRdyaBiQ4ZmiRKECAdgsZ/MSjYA1/jUQTtx5r3ZVRqJuSUDcz40ctq0tFXFylmyu1SgZ+1AX6I0XC/JjMhhfUUMUWsPbFT5THUW8xEm3JfXYsMLV18l6BXYWniv1+0dPUJoNn05+/PHHydHwad1VOu+x57LDA30kYQTaluhAHFLT4qPENwv/VYzwKwjkNKXLVLO/gIFY+ua7jLYqaKTFdgyqrhJBtjRZJpWIhZlYjQ9O1wr1i1MPlTolc6bAQjxYbcmg1P8mZQm77ODpkz8++fFffFCR164j1ASlCFuX6zm6SIt1CoIX1jRPv/BYOuyShjQlVLRFwVP+BYjVCztKyXjGOMZsMDDSbqJCpM2V65mwLIkCcVRFAUtCHpLq5lEMlPDg+ufMNeEDtDN73oZxfo/+F8b2hrVcdtXmjfmXZM7H56H40rsYGXa7vJSjtk/3z9n7jL389OpYuPxiTLHAc2AdEFpoOMAqCnn2/v07BrIiY9GiqnuignyKWui1MtyVMnZC0CM8R66KLQVpbZMaU8+ugN7AIV5aSLeiXC+73nr8p0avEaxSvLncQXf8h6Kh8YZ8JHFLAyh8xVvJG5YwtapLuFJqRC/YSV38B2yWohpfXHqR8KN2nIkUvnyl4s6wAWHdMYeqsVDw38sPn/yEc4kSzlfrbs9HEhPFiXkqRZKRaS2ptGeESAyuKE/zW8SShM48h602r4TwoZORIrlKwPNndEcF/R5CPLBAzfg8Wpd8V3OjYgHsGrY2WHIiogDaFWT0FQfL0AJJqba3QlywW3AAob3jC2NaeFRZnYGrF+JSKcWSoKEFriZnTaqFUKsUGyTvhjeb2HnCaHvDngBuo6vkBaezvzlmaWPKsNwqigaB0NeSpEGNJCMLrke+bkFY8M0A8wsjlK+VuKo+iNMBmOIDg9/BLM1nB/Nh/Ec+e/p4fvj02Q+zH4bxj3w4mx8+G87jOX+6WMSzZ8+ePY2eHQAzHZAX9yRcbR+9fTo8evn4cCD/tVDqYvp0lMmjbNId5MA43rKJpMi8e1OljCYLpErYRgX//syJnxV8Ca4cZrXGyBhIXAOb5JXcNRKHpeXaSfly4eyJz5i1+HnNu+5BGIqMz0ZgqDGaWvpc5SKDMCxXEbgtlJ2NjjIzflMvrHIhatCSdTdleKZ/m31XPz1qHDNhodi77ikrHmjZt+YbphrBIFvr4xrmtGy1tsR1CuP9w36a30RXBHKZiJClNwKrUwHCZSxkr3B/5fEa/CjWFLoqdwSZtwrzpinZNp8H2o62B0/2I5qBdatRmklNy9FF9t4nTIvO9q6TI9v12rrOpBwO9da2h8PV3Yb7EWPRkRCUscuBv1PUPnVvQbgE216Ld/A/e9U6y2hDccRJBvqt7NB2twJ5D0XWxrTh6aIQu+Fbtu8Ouu+1q5NKxInwkAcTD3IlSqJ1lS+jStyhamYC27PzOlloW9NTFr71MNO0nrXo4hoo5wi6yawhBNKIbtXSR7ARvn9CD2aowmZOiGqWRstZHGEqwMYKD/34eOhMSxlpIQZ0cAt0wNzpILttvTUPcrksj8tS8fL0/Av6HdLGzTP56Iv7ZsUOX0kedeNa/rJefgC+LYpoK2+t6fhk7WBcncn5AKa5uLohDMMAM8trqOF5WpmD3syh5BqcmLATNAB9Knnx74IqQctWOZ+Atl5tYYmkbRnSb0tjbnWLbUsLpXuNhhNp2BhGoX2HTNCVI4VnNZFmnTZOREQRQFESZleACJdJ1u2B4HILow0Uujkx8imZ5to3S/Bj61J/0N7bLVlYMsnbQmTatEE3UfxG517XlSDjTOZo9PuWAOo1+oWLRBMGEKOsp8EQc/MDtX6eTgSyhS/wg6uIaEwSynMTfhVNh0rn4F1XvaaYEW6FuNqJuw24Nk6UF0RhdzLOETr5YjQGbJbTfF3MPWLLWMa2xRLmxdUBWGIHKDYnXCbQHyB6B6s0ryZr2CtJSuSfCKsE01nx8tYj2v+6rDGgDIiF8+skjcG0hwVZVBeSEvQ2jltdYBqVrm9OgGhZ5/IwAg8ti7uCqQlgdY1rl6exhtVzF4TOpWQ/6eyWEyFOuo0Betr4NUDutDwvNmjM1nxmsGxR7OJv/LdHS0ceN8K4tC4W+w9tQeOkvnsbH9pO4+XZiHUN+eXb49PT3beQsccbNGQLUKHoJckoksAtzkGw9EHc9lkUx3i6L2/XkvuZZOiXm5vEIMj1XTuJE0nzdSnyYs70nbtiQJd7KKdTXtXbcTqtnUPcAOAIwcYSAQI149xOTj2zIi99hUcfxwIyBtph7Ws3UNdpEKA4kkrebFKOZ8lOwCSkDTlk3dfJFU5z2JOPYekbsBoGGmFEPhyQTFtyHfGmpH2ZzXtPpq/vyTQuykxlA31TpnFVBmWhEIDQWuFzS072dGoqp5qs6tTLPcklFemcnykS21BAb6BKoVSnHl6mooWeKS7vTVVqwVnO1uCVF3iQFTt5o0qgCFRB7JWg92FkzUsUgWseNetTOHWI9pouBuNFerzN2Z60Qvdev/B5lRfKVqH7Qgae4GYMkqR5flMyfee+HFlnhpsJBkdB7Q4pshTgSUN3Q4eASUkSBSUgiZUeOzClt0lcXQfsUPY6lFfM3LQrsf/I5FXx1pXInm5D3l7aBpoT8KrQoQFs+1RAyVigIf/Ki7zsWgs6kIY16EDRENpd+m7YjhSv6mfLNIUx3gC2dEYhqH4/Sm/Bruv3tYEmjX8QyIo/KbhKR9GKr8Z4tAp2P0acVa1JHqK9qYeREQ2JCtAKkdCgBgRJNxYmZA2VGb9KMnrBUEI6gaUQqUzQZKAy/iljHCejnBAiH+uSNNtfRSRP9kVkTEJeJnGc4nH6nhFObnwPaVQmVxkYN/MoA+e3zxZATjrar0cSYQZSZlegfFJM/MH0l9ucPKl8BqarvJ7nLM/OzYCHdXIZInpIxwgq2VpflhYIW+9PQA/g1OVK+gUmDVjFmD6UfB3nGC0dtaYndTc9YMxjk0L8ViUl9/Th+plzB1ixvh8/XN/aoxYmfVNS5ThFisuoXB/Zt2/xL9A94ylMB1a3dISgIhRJA7p/slVQIsCAlhVwcGiBl+zXKQg+nq9LDLDmPpB26jyInZUQwG7yjb6CLpLQVYq4qkMiyavgOwlkkJRMqrT9HKSxfsZT2E54Q705Ct1V/RuNIZIQSTQuo/KmmVpnD5xkdEE/Nzygz1JITzjZkHqIb4G7T5bGvhRB+7gx1Abvkb2kGmr7a9dTKHfmu6vlVd+bV91MztVMHLjTEQex+NQ+75j67xjJu9AqKxmPYE3Y1XMBSdc5VzV0d6taebJYtzFWgGXrqrYNw0HqGutVFvFqwFWCAnAyW+MVoW5HMHVHBZaF0moxehrPrdRgIeveF5IB5fKozgmUbkRmsn6Hu4d3VrRTA9sJGgPRUzhOkcj58Dy4cBcV3aHFeE6ZcEHFsGC3ci7zw/Tykxlrrm21pf7hx8EMP3gBKU27kkfGhpMwpcXlF88FHHkKnN+CUxrNb5qRiYtGCX7MYyne6vaOLklqhqdIlU5Kkza+TLJ1WY90+5EBupTdrtkq9D/lK4ommA3ca8alXLw+wYj0aIs2a7rSNCGe9kQYmrh4jD8bs2/AStuCAO7e1HjY9C+9Nc1AjUBoI8x9sU9V6FlpbJ1CLZMF2VWClmbrJV78OJzaaOEiZ83C2pzyJpKJ3O26hiRmYVybaqc1Bxxq5Tc2vZca2Ol3uy5TjwDx2wP6LYnKGCnoCpIloa01PFCvgbQ8cNK2m5U+QrEtbOi7EKk/aGokK3qYGyytCzx6vAxrQDFkIDw8styEc6TJmEZbtMRW6roqGZHKKaujB75AirMBu0OLxpCO/ChvRtk7wlrQnl3rLWhNZtAxJuyFkGvnsbfQYBgewvLoLi4gqB9oCOLyTVvbR7ucOM99VbF+Fwk+M4VXGm6di5C6Ga6taTRrNHrkOGzNkcudQ4+k6XGRXKK4EniMdiLi6TJrdnnEjsnt1m6fJWTJI5eCdsaFrK1rUTupo87p6Jvoe/vAWmS0EpTQvYtrpLK5iZvYi9H2EuyFfrTm73Sp9FuedVX9aw+mP6/rTPeFs3tqeTnUCLY5OaNIuy5sjRYlIpsPcd1FB52g0NKj3XqQsAbDkR47yRCcXx9JEhChGpZe2TSa7I9lB1E0rx0p/IiEm3bf6C5FTrx7nacY1qipm32zUzklDoHSTDHXEM3Dpkp04Yq4b7IZwBrRixUA5o4uXWDC9VLobCmwhd7U8S91ie0OQLT57Ax93JUq+ql0pki08bx1YX98C2jx+27iWjxz74Y15mrv5+c7TwfP2JhXMRi2nXk2n5lqieZ8w5mF960MTCUziVR3vSjVEikf1CLlVV5FaXt83K42EXIq9XkB/vC4DURmb94jQO6+BKvj48psEGcUCqNFGmFeH1pU1IxAQNNp22nSVIgARFjHKVVSGWWQqRtLv+8gxl0PpaCxSI+k7DhTsyIO/4CHL2rM4XnGAufgaHRMHeBpzfxTaoRatyoQUhpgNnTdtmGaw0bthciaPZmfgFf67VPPnc9Z2LMIfotHLBB139MSodwJ3cFRr50mgg4bTI52G1y2SrOdb+c15Vn9NdIdcsp9X26nwPI3rT2HN92xb53dYSR7cyO3PxXmfSWs/rCXqZGpNiP1h5AuRGpa57gD/3vRcS9A/xbiozGnu179qr1DZR6F8jxChZJUBJXxqWTxGpV1vOUC1qQRtxzpbVPUBvhMEr67WDsXMtdI1iBOoqZlJc9DCvUAGJoz7ptg4YNeoXr4XzNpoOT/8yaRfpMbGYpf8aL9D520WB/tORgisa89p/KuBDd0QacSgnn1XCVWqjw0yiaTrcb7x/s6GxGjcL5kNFLz0fyG5BqyJiyIeGL7jhRh6llw6AJ4xOIwD1OnAA4daypUvWDeZGyxFjHJfKkuzqL5oOclb9uW65SOKKOMiT/c1XygpedjXvyDbsede72ifse5QcvlCk3jF/v3eTWs9k7Yu8Y7YUmp/xrdC8BDT2OM0qgWMRfC/LChlu/9RxjE+1LWgL4ovTPk/Q0WD6b2g2EOwprarjFvdF8twlBrZ7/v2YxEEq0VHWGZ1Ksp4vXaKC14FG+FfMvkEbtN3Xq4KyrH4pazIVvgoVaNsvVQ5gsZc1GHgjqPDkcN8HqrKxnr8zL+rhFf0qMVPmvzjzLUY4reg3YnFSLJ5uKEmNQFJeRVLF4XIrtWBKZrL1DKd6QpQjSxHyyTFuL/ltff7ni7zX37rbXHkenxu3iDzXlt++EPU72mVNLb4EGPbFOUQz7P3XgLqWfnBWkhQiayMp6VEHRMZ7sGzXALikiC8hrpTqV6aQ7LvvG1ORuc/eKcC1K5BtQaV0BN1HUGVOkfRMu9/wZQSwMEFAAAAAgAG2loWdK83/prDwAASywAADUAAABydGRsX251bV9lbWJlZGRpbmdzLTAuMC4xMi5kaXN0LWluZm8vbGljZW5zZXMvTElDRU5TRd1aW5PbthV+z69ANdPp7gwtO07SNs6T4l0najdaz2pd19PpA0SCEmqSYAByteqv77ngRkm7cadv9XgSSyIODs7lO985oBDP/1n0stwpcaNL1Tn11TNP/k1Zp00nXs9fFeIvshulPYjXr159++Si3TD0b16+3O/3c0nbzI3dvmx4K/fyK1x4f333y1osVlfi7e3qanm/vF2txbvbO/FhfV2Iu+v3d7dXH97i1wU9dbVc398tf/yA35CAr+fiStW60wMo5+ZfeW1m/kQz4XayaUSrZCcGOOmgbOuE7CpRmq7iVaI2VoxOFcKq3ppqLPHrwovCZyvtBqs3I34vpBMVbqkqsTmItSpZyNcg35pxuxPfC1PDBw3PmXJsVTcc62XsiWKl6Q9Wb3eDMPtOWQEqwUI9HIQch52x+t+0n5dzbsWwk4OATbdWwsJuSw95O2QKqK1sxDWJPlFi7PCApL0SsiQpQQswAzzrxRh4wCuoleOtwaCDNU0hpFXhQ0NKF3ga/HbsKlhWmrY1nZfkHxR7PexYDm84F++MJT360fYGIiZZNTo8+GjmpczoKE5c6EteavbKFuA+C15CJXTH/y7EYEQpwen4nJfCP5EFrGhlJ7cKnYf7urHcecUKsd8pOj54n/aVJDu3zF5jNIGUCw2akHvcTvcoqdY1WLNXtkTRF9+9+v0lbWfAPGz4IGgc3ABWRx+Am6xyQSKI3KgOjFBqcOVEeqZncvknM87EBazFf9nZZe51+Is2edDViLKsyOPDC1CPoK12qAjo3WrnKOApzjgJyC0nobaG3UpIQUiv9jjSeqtqZS0sp19rsvhn3KI1lYajScqq4GDdlc1IpoAkFJ0ZRKNbjbuDH52phz2Gl6MNwSkVWD/kHgnyYviBIuR/rbejpd/BLY3K4ON28y8IhVPVZXfg78AdY0P5UVvTwo/lTnagdUgQiIrO4ZMyBBR90/iPtZCCzUPiiukBvYyjY0La9BoTypBy/phbiAQ4A3w9OXCOXnDSB0Zvh3I4d1tVaSmGQ58f+6Oxn09AYQ9fksaEQxhpKQV0F44RE4BN54/VygqA5EHqRm6akP8ZLhWIphiApfShJCMuBHQDM8DDEd7YUvCwJrPKYcDaQhYK2noRF3AA9SjbHnaGhQDtEOa8EJ9c9L2CnR8hmRqzv0xWuFJWP4AVH5RAg7jZcQTgHudt4E/vJbENguIb6dB5HaVihXtg9EP0MFbhVuQuzIX9Tpe7DAzAWQPUAMhMqx40uRKjGEzj80QosLCx4ROI8G7Os8kLwyqnHEQKWV/CZqahpIBleqs72OXU56d4HHCqnqR/IY7N562H0ex9R+J91bCqlTrmp+qlpUhBu9AxWmVVc4A86D6T4TYQLRgnnWzVZXC6BiCytSypSBRZjYxGPVEKraNMnbz+FqHc1/izHj/OgZiy2X7RgD7hQi2NeqCwiU8ohivPRIIkw7ahVfD7U8oXWVIMiPoGtm4CbLtxA9jhwSPwDoou0pzU86lAGxGOn9CK4GUqd89Wi5yoICrT9hjvGwXGrMEUT5OXL6v2YhbPNPOyuN5HWIZFqoEEtAbAuEAvbGRDcbS3uK4j8jF23voCsyA3ukqGQjsNLiUL2d8Vz5aiiF35HvA36QSIqBtc3AClBGlZyYpUyB3coFqXQzjU3FFhCSmpRvon2P1Y+ZitRK6VG73IYGQSBZm10W7AccvRUZWnHVvCS08jPxLipdKkHoMRpmcN8QhHcb0uRzM6SN5W2s8IfTaxo0C5lNPbjrAfQhF9RIY9G4kIVrMV2FuKPFfns9MUPuLX8dghA3+T8uQGRHxsjzYVO1BmoyCegDIqQnJQOt8nJaFTv44QPw1uWxqwN5drJLxZ+jEQvZ6Ln5BW4bZv4/EDsxLrkYurj9WzzUyWZjkqK6iSIjOQQAgBnYnFES8AcginBIbXqwEsE8IPoK+p9hq5Rme6F+R5ByfGjy+A9dgtNk7mIJvh8KK2Cj5pIHYPpkQgP6nmvv/DDUO3BSsgx3qM4xOkS3DejxtYC1aEQO0bCYEevwGdudQ6+sYTi7xvy2l+xGIiyyc7ninnhC3soG8yB72XCLr/B965gGWqHzDBoOUYAkUCBR03RJei57Nm3gO6DsJ28kERywsKUR9t6hp5HhQB1QD88n8BUYwd2DERBzxR9qyQYCacDE3APgq7yr5vsN00HTidrIzY5VUrG6nB3vxsdjiwIgnJrRtxs4PsdU5aTdlZW0Cf0NEoHWpfnvgX7hLaYNMpXxEB/oCRRFZPy44XhANxh+urLajPJG+qnN9ij64ItW4uljX6P/ZCDpAKYzo6ZdBbVkFuJf5MIOcb94tUsCK3tsa5F2QwPEZpRuRP/Bk8L0Uj927UAx61UVsuAmCxoHziBEeo+BzAUU1gxZ1vtZOcMjnnEI4V/NESUwUxTMWmkRgoU2hGfaaERiPlmC95gVVxdcAURe+FWJEuELYKvgzBF60L0rBPrBgKvp2LO5VPhua0dSsPCdmOUQhwUAduM8GjZ1geuQRpI2w2AshRHCGjgf+bWJGnbTOX8CeQrEitEBkkhVarFHu5Ng30RFzfA3a9CXX2Ql7ySUeItC3qi+pxvwFu1XBEBK2c+sbuEP+cHFRSfTjuJH6gMhr23GR78uAmUWnso7B/56GOxRCC9kF3GCfcPbpse4S4GNIoE1v3LRlDsZzpzmW2s1UDJFgReHPWwlN3ABodHy7bOG6YAqLADEvVsfDRXSAsVgp5U5GRCQrRIaWbPxuPIM7ocwyp+CcxN0bPIIOUqwwRWqgyeEw0J2ecHVLh4pOcluqp0apLBK3of9/4oatnq9v75dvrGSTf40D2xrTzeyDlzvbJsyuDgDOZcmJZ8lcmKrSeEnwoK+oxU9Cps2ZFUJI4583EeFAjZOCD0BGKL7FrJua8hc/alYINZDRKOmyn8im9X5KyFYgRbPomqCmDjsnWyUKTqHLP6vBDDuaTIMvzejqAErpOOIMlc5sq4Kl8Y4tTK8vA9bIpl+8NzlipPsoUIhDQAbKzQKCtXuAhD9E3Hc7noGFGYqEkNKH3O+7CEL9OzZz5m8gDt9JxyAc9RGpekaFM1fG5RYh1mMzmY9mQVYX/ttjv5BGZSQmqewt9SSYUbH0HjsjPRP0UjjeqSnXV2AbaOomYACzc/wV3HmMaGTgMMcAMZ5OJplXQMzEPsONx/LFhnrq3OGui1FUQbaVhPROAo8FX5goU4s+Rq4wjOY2sdcJyzzD4NNo7c2XEYrK7IlOf0aZIaVNTs3h4ohXJp3MxlUgebp1N85ICJ7dVkyocWTfOkolKYxxNxjKxUznqBCYO+Y6aHX8TwL1qYoFuLj50UEUdOU09wkalxvaXJGYXJHG+cThmkdkwKxtjPTm6Skwfdzwe5DDV2+TT5/+mNfM0i9TMAoZFMHWtwu0jr1+ZARfF2xuqLxvDTRmm7ZbaOywjpJoboRw4VSm+CMI0yFziN2J2wQNSsGJsibbQ01HgH3yGUEemHlWZQTwBbzSIVVtp+V7puPfwdwF/BCgMBMQhLGY8ujKEnANT7uxGCA3vL9SYvoRrDNni3CwyGpx6KfuAM33/EXTyMcwPh6ANGodISW2qVb+O2t8eYUF34BMs6eRSKPymxetp1AasDLyjhAN6V8SmAye1J/PZkE3Bb74anCkBbKk/zcWVdtQ64aVtLT4C/wS7HGISRFU3B25gqfPGFivBAHmRmpc0BSuSw3zuu6TqBeqKQ4PjFjV/GseXE+de4lwLIH+2WIvleiZ+XKyX62Dcj8v7n28/3IuPi7u7xep+eb0Wt3f5tfztO7FYfRJ/Xa6ugO5ovgF+xOmoSyfRhCtVNiZNGURzUhlw6gBNLpmKGiJ7CrFgzPvl/c11AVZfvViu3t0tVz9d/3K9ui/EL9d3b38GLRc/Lm+W958ohN4t71fXa359YOFlvF/cgcM+3CzuxPsPd+9v19dcbfm2sMGbBdC/h0013TrQzQx3hdNwAc9Z01uN9JwOXEN04SMUfwlxs3kpTxudA06Exw1wrR0huzOljm0yg7q/Z6VpbH7RetrMcuz9eQ6fg0lx0Y2WG93Q5fkSK68A+tMNpAfLgK8aGnaCjtBpZ6OWcJMFATTkI4NObRsN7KtUl0W87S4mo9w4+fnNeL9gooAz/UZviNCRclucR8R7i7DlgG8gOLodP58fjJ6T8oFDmeCyRtPGfiJArpWt3E5n+Lg6vBKQXg5wvcK79ez2GRIKiC1fJSCB4ZkuXsh5oQGhceYGeuO42vKdOVbxWKvx1vi40SVrjhFjRv5Gd96ZGa7mE4OLZ+/Eg1Z47MZwwG6Nqfa6yWeHn6Eom76XOCVETjCi4rXUzWi5GsmmHrtEbqgInnkTBG8BMHhze/DGykHgYBwiQT8exHkZcZguqwdNl6S1f30DMsAbIbzc4MVzBnw/F4sSawJaISAv7rxIhTpLio87pO7TdD2+LHz2ui2w0HJnDE9BadI5uWynmSvwtloRngDUkYayKxUfoucxqEe/A8Wdajt8tSQNxNisTdBdmE3jp1DEW14i7CDz5asWOA/mi++vdEDQ2GD8bPbYCXErGQ1G9swEp/PRGy1dk92GRM7tr0VoiOu/RiBNMEr6EtNJtygJ0dOkKAsDPxPGnknXjM+Y8JzvZJs62qZSNbQrvAKYcXVmdC5tS0gUyHW0Ykrn0dp0W+Ynx4DJ0JVjs8pD1OJ0brw5eLKRDnRACySbRjK/z6Ixo41RFw7g69UV1tVzr8HR74v37+GR5d/foAtpWgCIevCvL+Sv7uFvpMo+3iXBn/svXFD41yim04RAqw1kjYU2fAhTjSJ18rVWTeUEFAhIdgb9Dd5SKojM2T/+OYvAR5MJX+0OIZgIVX3Xl3XSc3FxZbo/xPcFshwNwn93KahbpzbVAb2ASACKH/Xw3UFWtrO7WcwVdwA8f4wXodTUswKAE7CwcXhBxU/7OWlAcXqW4waiDBkrt11EM/tQjMPV6kalV1bohjRo4nDhDJSjwTVi8AxrxfTm07/8gmpC4Ol4H+8tF+5d43gmDTmkLXd4Y83BkC4TX796/Y34BAqrR3Fz85Z+9TFRZS3SNFqK/P1PcYEPxFcsL39AEaH9wLznauWn5YG16853nYSEMYAioxGpyTcbGo7JyYQuxK0cQnT/1humN0DVV+vrF6AyLfkSQv4U1fCvmKGYbIJ2+kIT3hHkDzxFuP9Hth14NpltrdREhRDTxGIgROBo3XaE+AIGAFWgO36Rzw9HEj13p+eaf/UfUEsDBBQAAAAIAAAAIUgPfASzUwAAAFIAAAAqAAAAcnRkbF9udW1fZW1iZWRkaW5ncy0wLjAuMTIuZGlzdC1pbmZvL1dIRUVMC89ITc3RDUstKs7Mz7NSMNQz4HJPzUstSizJL7JSSMvJLFEw1jM0AgoH5eeX6HoW6waUFqXmZCZZKZQUlaZyhSSmWykUVBrr5uXnpeom5lVyAQBQSwMEFAAAAAgAAAAhSMdb9OTOAQAAhwMAAC0AAABydGRsX251bV9lbWJlZGRpbmdzLTAuMC4xMi5kaXN0LWluZm8vTUVUQURBVEG1Uklu20AQvPMV/QAutpVLBrEBQaIDIZRsSHaAnIwRp0VOMgszix3+Pk1qiw6BTz7NgqpeqmqJgQseePYdnZfWMLjJPyUrrpGBC0K9mKhfUG9RCGkan5xgV/lVfn2TbKLW3PUMHgyUJxjsrINV1OhkzRXcIw/RoQdp4Ilvo+IO5ogdVMidIUKefMP+zTrhGXAX5E7WknjSBFRKNmhqTMVAUAdCquTWUd+060NrzXBYV7cpNSEEXfbPsG92PGHYNJlGojgGP6Lr4at10rdG9skaf0dJ/OxxLMng7naSf05mintPAyExFjSPEShgGoUchgLGaJFXVLYjXd7HburxWqwPY14wKlmj8SPuYbOAadc5+0oF6D3teN0ibOwuvHGHR+gF/dHZxnGtSR2ouGkib8Za+3WG2+QC/2Q7WZ+GGjUvStNIg+Qa1Rjanq1Y/GPFf8ochzsIoqno8F+NTkly/zzM0oqo0CeHPbJ7qShv1WJWrjbl2Ym59IHBaOXd7TXFLf0ySWjPn1iH7HldMZhZgSm0IXSeFUUjQxu3eW110XMS/092zEMxZDmjLGfnLBfBIRaaS1OQvL9IrmJdTufLMtfissvc1nFYhweK/oe0+wtQSwMEFAAAAAgAAAAhSN/uhZsHAQAAswEAACsAAABydGRsX251bV9lbWJlZGRpbmdzLTAuMC4xMi5kaXN0LWluZm8vUkVDT1JEjc27doIwAADQ3W+JCAR5DB2oRNBilIhiXThgAkaeJWCVr+/k1MUfuLfraRnXQxWzKmWU8joXUvsE4pqoc/1jj2PbpcTTNjI/scHPD9Fo6Fz2sZ40DwL9SLsRZXcdBwRUy9LMSfffm8qSLCmqRLnop7zOmlnJL6wWTMz81QLhPXp1OMzTxA7Y3QlWWkp/RDGWRu9Bvd3eFS4Hdax2ZuokzTMAigKh9V4XeQj5r8NV8wMZQ9RXjJh1R0+HdYb5Jzx+Hx883wXTFAXDrcUcC2Cq7wUbFNqOHdqvY/xyq1+kLFfw2bXxGY7euYjmF6Y3xfK6DtPSCLXsvOh9KoAlw/cSghZb4gAw+QNQSwECFAMUAAAACAATco1agNpPeC4eAAD8dAAAFgAAAAAAAAAAAAAApIEAAAAAcnRkbF9udW1fZW1iZWRkaW5ncy5weVBLAQIUAxQAAAAIABtpaFnSvN/6aw8AAEssAAA1AAAAAAAAAAAAAACkgWIeAABydGRsX251bV9lbWJlZGRpbmdzLTAuMC4xMi5kaXN0LWluZm8vbGljZW5zZXMvTElDRU5TRVBLAQIUAxQAAAAIAAAAIUgPfASzUwAAAFIAAAAqAAAAAAAAAAAAAACkgSAuAABydGRsX251bV9lbWJlZGRpbmdzLTAuMC4xMi5kaXN0LWluZm8vV0hFRUxQSwECFAMUAAAACAAAACFIx1v05M4BAACHAwAALQAAAAAAAAAAAAAApIG7LgAAcnRkbF9udW1fZW1iZWRkaW5ncy0wLjAuMTIuZGlzdC1pbmZvL01FVEFEQVRBUEsBAhQDFAAAAAgAAAAhSN/uhZsHAQAAswEAACsAAAAAAAAAAAAAAKSB1DAAAHJ0ZGxfbnVtX2VtYmVkZGluZ3MtMC4wLjEyLmRpc3QtaW5mby9SRUNPUkRQSwUGAAAAAAUABQCzAQAAJDIAAAAA"
  },
  {
    "filename": "tabm-0.0.3-py3-none-any.whl",
    "sha256": "6cabed3cb436fef5e8a45f46097fd745f51b3e16be16d555b9af16cffda7f944",
    "base64": "..."
  }
]

if Path("/kaggle/input").is_dir():
    with tempfile.TemporaryDirectory() as directory:
        wheel_paths = []
        for wheel in WHEELS:
            payload = base64.b64decode(wheel["base64"], validate=True)
            if hashlib.sha256(payload).hexdigest() != wheel["sha256"]:
                raise ValueError(f"Offline wheel hash mismatch: {wheel['filename']}")
            path = Path(directory) / wheel["filename"]
            path.write_bytes(payload)
            wheel_paths.append(str(path))
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
             "--disable-pip-version-check", *wheel_paths],
            check=True,
        )
for package, expected in [("tabm", "0.0.3"), ("rtdl_num_embeddings", "0.0.12")]:
    if version(package) != expected:
        raise RuntimeError(f"Expected {package}=={expected}, found {version(package)}")
print("Pinned offline TabM dependencies verified")


In [ ]:
from importlib.metadata import version
import json
from pathlib import Path
import shutil
import subprocess
import sys

expected = {'torch': '2.11.0+cu130', 'tabm': '0.0.3', 'rtdl_num_embeddings': '0.0.12', 'numpy': '2.1.3', 'polars': '1.35.2', 'scikit-learn': '1.6.1'}
actual = {name: version(name) for name in expected}
if actual != expected:
    raise RuntimeError(f"Pinned training environment changed: {actual} != {expected}")
sources = {'train.py': 'import argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nfrom catboost import CatBoostClassifier\nfrom catboost.utils import get_gpu_device_count\nimport numpy as np\nimport polars as pl\nimport pyarrow.parquet as pq\nfrom sklearn.metrics import roc_auc_score\n\n\nROOT = Path(__file__).resolve().parent\nON_KAGGLE = Path("/kaggle/input").is_dir()\nTARGET = "watched"\nTIME = "feature1"\n\n\ndef features(frame, expand_embeddings=False):\n    expressions = []\n    for name, dtype in frame.schema.items():\n        if name in {TARGET, "request_id", "id"}:\n            continue\n        column = pl.col(name)\n        if isinstance(dtype, pl.Datetime):\n            expressions.extend(\n                [\n                    column.dt.hour().cast(pl.Float32).alias(f"{name}_hour"),\n                    column.dt.weekday().cast(pl.Float32).alias(f"{name}_weekday"),\n                ]\n            )\n        elif isinstance(dtype, pl.List):\n            expressions.extend(\n                [\n                    column.list.len().cast(pl.Float32).alias(f"{name}_length"),\n                    column.list.mean().cast(pl.Float32).alias(f"{name}_mean"),\n                    column.list.std().cast(pl.Float32).alias(f"{name}_std"),\n                    column.list.min().cast(pl.Float32).alias(f"{name}_min"),\n                    column.list.max().cast(pl.Float32).alias(f"{name}_max"),\n                ]\n            )\n            if expand_embeddings:\n                invalid = frame.select(\n                    (column.is_not_null() & (column.list.len() != 32)).any()\n                ).item()\n                if invalid:\n                    raise ValueError(f"{name} must contain 32-element vectors")\n                expressions.extend(\n                    column.list.get(index).cast(pl.Float32).alias(f"{name}_dim_{index}")\n                    for index in range(32)\n                )\n        elif dtype == pl.String:\n            expressions.append(column.fill_null("__MISSING__").alias(name))\n        elif dtype.is_numeric():\n            expressions.append(column.cast(pl.Float32).alias(name))\n        else:\n            raise TypeError(f"Unsupported feature {name}: {dtype}")\n    return frame.select(expressions).with_columns(\n        pl.col(pl.Float32).fill_nan(None)\n    )\n\n\ndef load_sample(path, maximum_rows, seed, expand_embeddings=False):\n    parquet = pq.ParquetFile(path)\n    total = parquet.metadata.num_rows\n    rng = np.random.default_rng(seed)\n    indices = np.sort(rng.choice(total, min(maximum_rows, total), replace=False))\n    frames = []\n    offset = 0\n    for batch in parquet.iter_batches(batch_size=8192):\n        end = offset + batch.num_rows\n        selected = indices[(indices >= offset) & (indices < end)] - offset\n        if len(selected):\n            raw = pl.from_arrow(batch.take(selected))\n            frames.append(\n                features(raw, expand_embeddings).with_columns(\n                    raw[TARGET], raw[TIME].alias("_time")\n                )\n            )\n        offset = end\n    return pl.concat(frames)\n\n\ndef model(iterations, threads, task_type):\n    return CatBoostClassifier(\n        iterations=iterations,\n        depth=6,\n        learning_rate=0.06,\n        loss_function="Logloss",\n        eval_metric="AUC",\n        random_seed=42,\n        thread_count=threads,\n        task_type=task_type,\n        max_ctr_complexity=1,\n        allow_writing_files=False,\n        verbose=50,\n    )\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--rows", type=int, default=1_600_000 if ON_KAGGLE else 150_000)\n    parser.add_argument("--iterations", type=int, default=400)\n    parser.add_argument("--threads", type=int, default=4)\n    parser.add_argument("--expand-embeddings", action="store_true")\n    parser.add_argument("--validation-only", action="store_true")\n    parser.add_argument(\n        "--task-type", choices=["CPU", "GPU"], default="GPU" if ON_KAGGLE else "CPU"\n    )\n    parser.add_argument("--data-dir", type=Path)\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("/kaggle/working") if ON_KAGGLE else ROOT / "artifacts",\n    )\n    args = parser.parse_args()\n    if min(args.rows, args.iterations, args.threads) <= 0:\n        parser.error("rows, iterations, and threads must be positive")\n\n    if args.task_type == "GPU" and get_gpu_device_count() == 0:\n        raise RuntimeError("GPU training requested, but no CUDA GPU is available")\n    data = args.data_dir\n    if data is None and ON_KAGGLE:\n        candidates = [\n            path.parent\n            for path in Path("/kaggle/input").rglob("train.parquet")\n            if (path.parent / "test.parquet").is_file()\n            and (path.parent / "sample_submission.csv").is_file()\n        ]\n        if len(candidates) != 1:\n            raise RuntimeError(f"Expected one ShareChat dataset, found {candidates}")\n        data = candidates[0]\n    elif data is None:\n        data = ROOT / "data"\n    output = args.output_dir\n    output.mkdir(parents=True, exist_ok=True)\n    print(f"Compute: {args.task_type}; input: {data}; output: {output}", flush=True)\n    sample = load_sample(\n        data / "train.parquet", args.rows, 42, args.expand_embeddings\n    ).sort("_time")\n    if sample.height < 2:\n        raise ValueError("Chronological validation requires at least two rows")\n    if sample["_time"].null_count():\n        raise ValueError("Missing timestamps prevent chronological validation")\n    cutoff = sample["_time"][int(sample.height * 0.8)]\n    train = sample.filter(pl.col("_time") < cutoff)\n    valid = sample.filter(pl.col("_time") >= cutoff)\n    if train.is_empty() or valid.is_empty():\n        raise ValueError(\n            f"Timestamp cutoff {cutoff} cannot produce nonempty chronological splits"\n        )\n    for name, part in [("train", train), ("validation", valid)]:\n        if part[TARGET].n_unique() != 2:\n            raise ValueError(f"{name} must contain both target classes")\n    columns = [name for name in sample.columns if name not in {TARGET, "_time"}]\n    categories = [name for name in columns if sample.schema[name] == pl.String]\n    x_train = train.select(columns).to_pandas()\n    x_valid = valid.select(columns).to_pandas()\n    y_train = train[TARGET].to_numpy()\n    y_valid = valid[TARGET].to_numpy()\n    sample_rows, training_rows, validation_rows = sample.height, train.height, valid.height\n    print(\n        f"Training: {train.height:,}; validation: {valid.height:,}; "\n        f"features: {len(columns)}; cutoff: {cutoff}",\n        flush=True,\n    )\n    if args.validation_only:\n        del sample, train, valid\n        gc.collect()\n    estimator = model(args.iterations, args.threads, args.task_type)\n    fit_started = time.monotonic()\n    estimator.fit(\n        x_train,\n        y_train,\n        cat_features=categories,\n        eval_set=(x_valid, y_valid),\n        early_stopping_rounds=60,\n    )\n    fit_seconds = time.monotonic() - fit_started\n    auc = roc_auc_score(\n        y_valid, estimator.predict_proba(x_valid)[:, 1]\n    )\n    best_iterations = estimator.tree_count_\n    metrics = {\n        "validation_auc": auc,\n        "validation": "chronological 80/20 split; timestamp ties kept together",\n        "cutoff": str(cutoff),\n        "sample_rows": sample_rows,\n        "training_rows": training_rows,\n        "validation_rows": validation_rows,\n        "iterations": best_iterations,\n        "max_iterations": args.iterations,\n        "expand_embeddings": args.expand_embeddings,\n        "validation_only": args.validation_only,\n        "fit_seconds": fit_seconds,\n        "task_type": args.task_type,\n        "features": columns,\n        "categorical_features": categories,\n    }\n    estimator.save_model(str(output / "validation.cbm"))\n    (output / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\\n")\n    print(f"Validation ROC AUC: {auc:.6f}", flush=True)\n    if args.validation_only:\n        print("Validation-only run: skipping full-data refit and test predictions", flush=True)\n        return\n    del estimator, x_train, x_valid, train, valid\n    gc.collect()\n\n    print("Refitting on the full sample", flush=True)\n    estimator = model(best_iterations, args.threads, args.task_type)\n    estimator.fit(\n        sample.select(columns).to_pandas(),\n        sample[TARGET].to_numpy(),\n        cat_features=categories,\n    )\n    estimator.save_model(str(output / "baseline.cbm"))\n    del sample\n    gc.collect()\n\n    submission = pl.read_csv(data / "sample_submission.csv")\n    if submission.columns != ["id", TARGET]:\n        raise ValueError(f"Unexpected submission schema: {submission.columns}")\n    test = pq.ParquetFile(data / "test.parquet")\n    if test.metadata.num_rows != submission.height:\n        raise ValueError("Test and submission row counts differ")\n    predictions = []\n    offset = 0\n    for batch in test.iter_batches(batch_size=8192):\n        raw = pl.from_arrow(batch)\n        if "id" not in raw.columns:\n            raise ValueError("Test data has no id column for submission alignment")\n        expected = submission["id"].slice(offset, raw.height)\n        if raw["id"].to_list() != expected.to_list():\n            raise ValueError("Test ids do not match sample submission order")\n        predictions.append(\n            estimator.predict_proba(\n                features(raw, args.expand_embeddings).select(columns).to_pandas()\n            )[:, 1]\n        )\n        offset += raw.height\n    probabilities = np.concatenate(predictions)\n    if not np.isfinite(probabilities).all() or not (\n        (probabilities >= 0) & (probabilities <= 1)\n    ).all():\n        raise ValueError("Invalid predicted probabilities")\n    submission.with_columns(pl.Series(TARGET, probabilities)).write_csv(\n        output / "submission.csv"\n    )\n    print(f"Saved {submission.height:,} predictions to {output / \'submission.csv\'}")\n\n\nif __name__ == "__main__":\n    main()\n', 'tabm_experiment.py': 'import argparse\nimport gc\nimport hashlib\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nimport time\n\nimport joblib\nimport numpy as np\nimport polars as pl\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.metrics import log_loss, roc_auc_score\nfrom sklearn.preprocessing import OrdinalEncoder, StandardScaler\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom train import ROOT, TARGET, load_sample\n\n\ndef prepare_features(train, valid):\n    columns = [name for name in train.columns if name not in {TARGET, "_time"}]\n    numeric = [name for name in columns if train.schema[name].is_numeric()]\n    categorical = [name for name in columns if train.schema[name] == pl.String]\n    if len(numeric) + len(categorical) != len(columns):\n        raise TypeError("The engineered sample must contain only numeric/string features")\n    imputer = SimpleImputer(\n        strategy="median", keep_empty_features=True, add_indicator=True, copy=False\n    )\n    scaler = StandardScaler(copy=False)\n    transformed, infinity_counts = [], []\n    for label, frame in [("training", train), ("validation", valid)]:\n        array = np.array(frame.select(numeric).to_numpy(), dtype=np.float32, order="C")\n        count = int(np.isinf(array).sum())\n        infinity_counts.append(count)\n        if count:\n            print(f"Treating {count} infinite {label} values as missing", flush=True)\n            array[np.isinf(array)] = np.nan\n        if label == "training":\n            array = imputer.fit_transform(array)\n            array = scaler.fit_transform(array)\n        else:\n            array = imputer.transform(array)\n            array = scaler.transform(array)\n        transformed.append(array)\n    x_train, x_valid = transformed\n    for array in [x_train, x_valid]:\n        np.clip(array, -10, 10, out=array)\n        if not np.isfinite(array).all():\n            raise ValueError("Nonfinite numerical features after preprocessing")\n    encoder = OrdinalEncoder(\n        handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.int64\n    )\n    categories = [\n        frame.select(categorical).fill_null("__MISSING__").to_numpy()\n        for frame in [train, valid]\n    ]\n    c_train = encoder.fit_transform(categories[0]) + 1\n    c_valid = encoder.transform(categories[1]) + 1\n    cardinalities = [len(values) + 1 for values in encoder.categories_]\n    preprocessing = {\n        "numeric_columns": numeric,\n        "categorical_columns": categorical,\n        "imputer": imputer,\n        "scaler": scaler,\n        "encoder": encoder,\n        "cardinalities": cardinalities,\n        "infinity_counts": infinity_counts,\n        "validation_unknown_category_rates": {\n            name: float((c_valid[:, index] == 0).mean())\n            for index, name in enumerate(categorical)\n        },\n    }\n    return (\n        torch.from_numpy(np.ascontiguousarray(x_train, dtype=np.float32)),\n        torch.from_numpy(np.ascontiguousarray(c_train)),\n        torch.from_numpy(np.ascontiguousarray(x_valid, dtype=np.float32)),\n        torch.from_numpy(np.ascontiguousarray(c_valid)),\n        preprocessing,\n    )\n\n\ndef transform_features(frame, state):\n    numeric = np.array(\n        frame.select(state["numeric_columns"]).to_numpy(),\n        dtype=np.float32,\n        order="C",\n    )\n    infinities = np.isinf(numeric)\n    if infinities.any():\n        print(f"Treating {int(infinities.sum())} infinite inference values as missing", flush=True)\n        numeric[infinities] = np.nan\n    numeric = state["scaler"].transform(state["imputer"].transform(numeric))\n    np.clip(numeric, -10, 10, out=numeric)\n    if not np.isfinite(numeric).all():\n        raise ValueError("Nonfinite numerical features after inference preprocessing")\n    categories = (\n        frame.select(state["categorical_columns"]).fill_null("__MISSING__").to_numpy()\n    )\n    encoded = state["encoder"].transform(categories) + 1\n    return (\n        torch.from_numpy(np.ascontiguousarray(numeric, dtype=np.float32)),\n        torch.from_numpy(np.ascontiguousarray(encoded, dtype=np.int64)),\n    )\n\n\nclass TabMClassifier(nn.Module):\n    def __init__(\n        self, n_numeric, cardinalities, members=8, width=128, blocks=2, dropout=0.1,\n        category_dropout=0.0, positive_rate=None,\n        embedding_dimensions=None, numeric_dropout=0.0, dropout_columns=None,\n    ):\n        super().__init__()\n        from tabm import TabM\n\n        if not 0 <= category_dropout <= 1:\n            raise ValueError("category_dropout must be between zero and one")\n        self.category_dropout = category_dropout\n        self.numeric_dropout = numeric_dropout\n        self.dropout_columns = dropout_columns or []\n        dimensions = embedding_dimensions or [8] * len(cardinalities)\n        if len(dimensions) != len(cardinalities) or any(d <= 0 for d in dimensions):\n            raise ValueError("Expected one positive embedding dimension per category")\n        self.embeddings = nn.ModuleList(\n            nn.Embedding(cardinality, dimension, padding_idx=0)\n            for cardinality, dimension in zip(cardinalities, dimensions)\n        )\n        for embedding in self.embeddings:\n            nn.init.normal_(embedding.weight, std=0.02)\n            with torch.no_grad():\n                embedding.weight[0].zero_()\n        self.backbone = TabM.make(\n            n_num_features=n_numeric + sum(dimensions),\n            d_out=1,\n            k=members,\n            n_blocks=blocks,\n            d_block=width,\n            dropout=dropout,\n        )\n        if positive_rate is not None:\n            if not 0 < positive_rate < 1:\n                raise ValueError("positive_rate must be strictly between zero and one")\n            with torch.no_grad():\n                self.backbone.output.bias.fill_(\n                    float(np.log(positive_rate / (1 - positive_rate)))\n                )\n\n    def forward(self, numeric, categorical):\n        if self.training and self.numeric_dropout:\n            numeric = numeric.clone()\n            numeric[:, self.dropout_columns] = F.dropout(\n                numeric[:, self.dropout_columns], p=self.numeric_dropout, training=True\n            )\n        if self.training and self.category_dropout:\n            categorical = categorical.masked_fill(\n                torch.rand_like(categorical, dtype=torch.float32) < self.category_dropout,\n                0,\n            )\n        embedded = [\n            layer(categorical[:, index])\n            for index, layer in enumerate(self.embeddings)\n        ]\n        return self.backbone(torch.cat([numeric, *embedded], dim=1)).squeeze(-1)\n\n\n@torch.inference_mode()\ndef predict(model, numeric, categorical, device, batch_size):\n    model.eval()\n    probabilities = []\n    for start in range(0, len(numeric), batch_size):\n        logits = model(\n            numeric[start : start + batch_size].to(device),\n            categorical[start : start + batch_size].to(device),\n        )\n        probabilities.append(logits.sigmoid().mean(dim=1).cpu().numpy())\n    return np.concatenate(probabilities)\n\n\ndef fit_model(\n    model, model_config, optimizer, x_train, c_train, y_train, x_valid, c_valid, y_valid,\n    *, device, output, epochs, patience, batch_size,\n):\n    generator = torch.Generator().manual_seed(42)\n    best_auc, best_epoch, stale = -np.inf, 0, 0\n    history = []\n    training_started = time.monotonic()\n    for epoch in range(1, epochs + 1):\n        epoch_started = time.monotonic()\n        model.train()\n        order = torch.randperm(len(y_train), generator=generator)\n        loss_sum = 0.0\n        for indices in order.split(batch_size):\n            optimizer.zero_grad(set_to_none=True)\n            logits = model(x_train[indices].to(device), c_train[indices].to(device))\n            targets = y_train[indices].to(device).unsqueeze(1).expand_as(logits)\n            loss = F.binary_cross_entropy_with_logits(logits, targets)\n            if not torch.isfinite(loss).item():\n                raise ValueError(f"Nonfinite training loss at epoch {epoch}")\n            loss.backward()\n            optimizer.step()\n            loss_sum += loss.item() * len(indices)\n        probabilities = predict(model, x_valid, c_valid, device, batch_size)\n        auc = float(roc_auc_score(y_valid, probabilities))\n        record = {\n            "epoch": epoch,\n            "training_loss": loss_sum / len(y_train),\n            "validation_auc": auc,\n            "seconds": time.monotonic() - epoch_started,\n        }\n        history.append(record)\n        print(json.dumps(record), flush=True)\n        if auc > best_auc:\n            best_auc, best_epoch, stale = auc, epoch, 0\n            torch.save(\n                {\n                    "model_config": model_config,\n                    "state_dict": {\n                        key: value.detach().cpu().clone()\n                        for key, value in model.state_dict().items()\n                    },\n                },\n                output / "best_model.pt",\n            )\n        else:\n            stale += 1\n            if stale >= patience:\n                break\n    training_seconds = time.monotonic() - training_started\n    checkpoint = torch.load(output / "best_model.pt", map_location="cpu", weights_only=True)\n    model.load_state_dict(checkpoint["state_dict"])\n    probabilities = predict(model, x_valid, c_valid, device, batch_size)\n    restored_auc = float(roc_auc_score(y_valid, probabilities))\n    if abs(restored_auc - best_auc) > 1e-5:\n        raise ValueError("Reloaded checkpoint does not reproduce the selected validation AUC")\n    np.savez_compressed(\n        output / "validation_predictions.npz", y_true=y_valid, probabilities=probabilities\n    )\n    return {\n        "validation_auc": restored_auc,\n        "best_epoch": best_epoch,\n        "epochs_run": len(history),\n        "training_seconds": training_seconds,\n        "history": history,\n    }\n\n\ndef fit_model_steps(\n    model, model_config, optimizer, x_train, c_train, y_train, x_valid, c_valid, y_valid,\n    *, device, output, epochs=30, patience=5, batch_size=1024, validation_interval=300,\n    seed=42, scheduler=None,\n    fixed_budget=False,\n):\n    if min(epochs, patience, batch_size, validation_interval) <= 0:\n        raise ValueError("Training budgets and validation interval must be positive")\n    output.mkdir(parents=True, exist_ok=True)\n    generator = torch.Generator().manual_seed(seed)\n    probe_indices = torch.randperm(\n        len(y_train), generator=torch.Generator().manual_seed(42)\n    )[: min(16384, len(y_train))]\n    steps_per_epoch = (len(y_train) + batch_size - 1) // batch_size\n    best_auc, best_step, global_step = -np.inf, 0, 0\n    history, checks = [], []\n    started = time.monotonic()\n    if device == "cuda":\n        torch.cuda.reset_peak_memory_stats()\n    for epoch in range(1, epochs + 1):\n        model.train()\n        loss_sum, rows_seen, stop = 0.0, 0, False\n        order = torch.randperm(len(y_train), generator=generator)\n        for batch_index, indices in enumerate(order.split(batch_size), 1):\n            optimizer.zero_grad(set_to_none=True)\n            logits = model(x_train[indices].to(device), c_train[indices].to(device))\n            targets = y_train[indices].to(device).unsqueeze(1).expand_as(logits)\n            loss = F.binary_cross_entropy_with_logits(logits, targets)\n            if not torch.isfinite(loss).item():\n                raise ValueError(f"Nonfinite loss at epoch {epoch}, step {global_step}")\n            loss.backward()\n            optimizer.step()\n            if scheduler is not None:\n                scheduler.step()\n            global_step += 1\n            loss_sum += loss.item() * len(indices)\n            rows_seen += len(indices)\n            expired = global_step - best_step >= patience * steps_per_epoch\n            if fixed_budget:\n                if epoch != epochs or batch_index != steps_per_epoch:\n                    continue\n            elif (\n                global_step % validation_interval != 0\n                and batch_index != steps_per_epoch and not expired\n            ):\n                continue\n            probabilities = predict(model, x_valid, c_valid, device, batch_size)\n            auc = float(roc_auc_score(y_valid, probabilities))\n            probe = predict(\n                model, x_train[probe_indices], c_train[probe_indices], device, batch_size\n            )\n            probe_y = y_train[probe_indices].numpy()\n            record = {\n                "epoch": epoch,\n                "global_step": global_step,\n                "fractional_epoch": epoch - 1 + batch_index / steps_per_epoch,\n                "validation_auc": auc,\n                "validation_logloss": float(log_loss(y_valid, probabilities, labels=[0, 1])),\n                "training_probe_auc": float(roc_auc_score(probe_y, probe)),\n                "training_probe_logloss": float(log_loss(probe_y, probe, labels=[0, 1])),\n                "learning_rate": optimizer.param_groups[0]["lr"],\n                "seconds": time.monotonic() - started,\n            }\n            checks.append(record)\n            print(json.dumps(record), flush=True)\n            if auc > best_auc:\n                best_auc, best_step = auc, global_step\n                torch.save(\n                    {\n                        "model_config": model_config,\n                        "state_dict": {\n                            key: value.detach().cpu().clone()\n                            for key, value in model.state_dict().items()\n                        },\n                        "selection": record,\n                    },\n                    output / "best_model.pt",\n                )\n            model.train()\n            if global_step - best_step >= patience * steps_per_epoch:\n                stop = True\n                break\n        history.append({\n            "epoch": epoch, "training_loss": loss_sum / rows_seen,\n            "last_validation_auc": checks[-1]["validation_auc"] if checks else None,\n        })\n        if stop:\n            break\n    checkpoint = torch.load(output / "best_model.pt", map_location="cpu", weights_only=True)\n    model.load_state_dict(checkpoint["state_dict"])\n    probabilities = predict(model, x_valid, c_valid, device, batch_size)\n    auc = float(roc_auc_score(y_valid, probabilities))\n    if abs(auc - best_auc) > 1e-5:\n        raise ValueError("Fine checkpoint reload does not reproduce its validation AUC")\n    np.savez_compressed(\n        output / "validation_predictions.npz", y_true=y_valid, probabilities=probabilities\n    )\n    return {\n        "validation_auc": auc,\n        "validation_logloss": float(log_loss(y_valid, probabilities, labels=[0, 1])),\n        "best_epoch": checkpoint["selection"]["epoch"],\n        "best_global_step": best_step,\n        "best_fractional_epoch": checkpoint["selection"]["fractional_epoch"],\n        "optimizer_steps": global_step, "epochs_run": len(history),\n        "training_seconds": time.monotonic() - started,\n        "peak_cuda_bytes": torch.cuda.max_memory_allocated() if device == "cuda" else None,\n        "history": history, "checkpoint_history": checks,\n        "selection_mode": "fixed_budget" if fixed_budget else "validation_auc",\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--rows", type=int, default=150_000)\n    parser.add_argument("--sample-path", type=Path)\n    parser.add_argument("--epochs", type=int, default=20)\n    parser.add_argument("--patience", type=int, default=5)\n    parser.add_argument("--batch-size", type=int, default=1024)\n    parser.add_argument("--members", type=int, default=8)\n    parser.add_argument("--width", type=int, default=128)\n    parser.add_argument("--blocks", type=int, default=2)\n    parser.add_argument("--threads", type=int, default=4)\n    parser.add_argument("--device", choices=["auto", "cpu", "mps", "cuda"], default="auto")\n    parser.add_argument("--output-dir", type=Path, default=ROOT / "artifacts/local-tabm-v1")\n    args = parser.parse_args()\n    if min(\n        args.rows, args.epochs, args.patience, args.batch_size,\n        args.members, args.width, args.blocks, args.threads,\n    ) <= 0:\n        parser.error("All numerical arguments must be positive")\n    device = args.device\n    if device == "auto":\n        device = (\n            "cuda" if torch.cuda.is_available()\n            else "mps" if torch.backends.mps.is_available()\n            else "cpu"\n        )\n    if device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA requested but unavailable")\n    if device == "mps" and not torch.backends.mps.is_available():\n        raise RuntimeError("Apple MPS requested but unavailable")\n    torch.set_num_threads(args.threads)\n    torch.manual_seed(42)\n    started = time.monotonic()\n    sample_hash = None\n    if args.sample_path:\n        sample = pl.read_parquet(args.sample_path)\n        if sample.height != args.rows:\n            raise ValueError("Cached sample row count does not match --rows")\n        with args.sample_path.open("rb") as stream:\n            sample_hash = hashlib.file_digest(stream, "sha256").hexdigest()\n    else:\n        sample = load_sample(ROOT / "data/train.parquet", args.rows, 42)\n    sample = sample.select(\n        name for name in sample.columns if "_dim_" not in name\n    ).sort("_time")\n    if sample.height < 2 or sample["_time"].null_count():\n        raise ValueError("Chronological validation requires nonmissing timestamps")\n    cutoff = sample["_time"][int(sample.height * 0.8)]\n    train = sample.filter(pl.col("_time") < cutoff)\n    valid = sample.filter(pl.col("_time") >= cutoff)\n    for label, frame in [("training", train), ("validation", valid)]:\n        if set(frame[TARGET].unique().to_list()) != {0, 1}:\n            raise ValueError(f"{label} must contain both binary target classes")\n    training_rows, validation_rows = train.height, valid.height\n    y_train = torch.tensor(train[TARGET].to_numpy().copy(), dtype=torch.float32)\n    y_valid = valid[TARGET].to_numpy().copy()\n    x_train, c_train, x_valid, c_valid, preprocessing = prepare_features(train, valid)\n    feature_count = sample.width - 2\n    sample_rows = sample.height\n    del sample, train, valid\n    gc.collect()\n    output = args.output_dir\n    output.mkdir(parents=True, exist_ok=True)\n    joblib.dump(preprocessing, output / "preprocessing.joblib")\n    model_config = {\n        "n_numeric": x_train.shape[1],\n        "cardinalities": preprocessing["cardinalities"],\n        "members": args.members,\n        "width": args.width,\n        "blocks": args.blocks,\n        "dropout": 0.1,\n    }\n    model = TabMClassifier(**model_config).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.0003)\n    preparation_seconds = time.monotonic() - started\n    print(\n        f"Device: {device}; sample: {sample_rows:,}; train: {training_rows:,}; "\n        f"valid: {validation_rows:,}; features: {feature_count}; "\n        f"cutoff: {cutoff}; preparation: {preparation_seconds:.1f}s",\n        flush=True,\n    )\n    fitted = fit_model(\n        model, model_config, optimizer, x_train, c_train, y_train, x_valid, c_valid, y_valid,\n        device=device, output=output, epochs=args.epochs, patience=args.patience,\n        batch_size=args.batch_size,\n    )\n    report = {\n        "algorithm": "TabM with learned categorical embeddings",\n        **fitted,\n        "seed": 42,\n        "sample_rows": sample_rows,\n        "sample_sha256": sample_hash,\n        "training_rows": training_rows,\n        "validation_rows": validation_rows,\n        "cutoff": str(cutoff),\n        "input_features": feature_count,\n        "numerical_features": len(preprocessing["numeric_columns"]),\n        "numerical_features_after_imputation": x_train.shape[1],\n        "categorical_features": len(preprocessing["categorical_columns"]),\n        "validation_unknown_category_rates": preprocessing["validation_unknown_category_rates"],\n        "infinite_input_counts_train_valid": preprocessing["infinity_counts"],\n        "model_config": model_config,\n        "optimizer": {"name": "AdamW", "learning_rate": 0.002, "weight_decay": 0.0003},\n        "max_epochs": args.epochs,\n        "patience": args.patience,\n        "batch_size": args.batch_size,\n        "device": device,\n        "threads": args.threads,\n        "preparation_seconds": preparation_seconds,\n        "versions": {\n            name: version(name)\n            for name in ["torch", "tabm", "numpy", "polars", "scikit-learn"]\n        },\n        "competition_submission": False,\n    }\n    (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n    print(\n        f"Best validation ROC AUC: {fitted[\'validation_auc\']:.6f}; "\n        f"epoch: {fitted[\'best_epoch\']}; training: {fitted[\'training_seconds\']:.1f}s; "\n        "no submission performed",\n        flush=True,\n    )\n\n\nif __name__ == "__main__":\n    main()\n', 'custom_network.py': 'import argparse\nimport gc\nimport hashlib\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nimport time\n\nimport joblib\nimport numpy as np\nimport polars as pl\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom tabm_experiment import fit_model, prepare_features, transform_features\nfrom train import ROOT, TARGET, load_sample\n\n\ndef pack_vectors(frame, fields):\n    packed, missing = [], {}\n    for field in fields:\n        columns = [f"{field}_dim_{index}" for index in range(32)]\n        raw = np.array(frame.select(columns).to_numpy(), dtype=np.float32, order="C")\n        valid = np.isfinite(raw).all(axis=1)\n        missing[field] = int((~valid).sum())\n        raw[~valid] = 0\n        norm = np.linalg.norm(raw.astype(np.float64), axis=1)\n        unit = (raw / np.maximum(norm, 1e-8)[:, None]).astype(np.float32)\n        log_norm = np.clip(np.log1p(norm), 0, 20).astype(np.float32)\n        packed.append(\n            np.concatenate([unit, log_norm[:, None], valid[:, None]], axis=1)\n        )\n    return torch.from_numpy(np.stack(packed, axis=1).astype(np.float32)), missing\n\n\ndef append_vectors(frame, numeric, fields):\n    packed = torch.empty(\n        (len(numeric), numeric.shape[1] + 34 * len(fields)), dtype=torch.float32\n    )\n    packed[:, : numeric.shape[1]] = numeric\n    missing = {}\n    for index, field in enumerate(fields):\n        vector, counts = pack_vectors(frame, [field])\n        start = numeric.shape[1] + 34 * index\n        packed[:, start : start + 34] = vector[:, 0]\n        missing.update(counts)\n    return packed, missing\n\n\ndef prepare_custom_features(train, valid, min_category_count):\n    dimension_columns = [name for name in train.columns if "_dim_" in name]\n    fields = sorted(\n        {name.rsplit("_dim_", 1)[0] for name in dimension_columns},\n        key=lambda name: int(name.removeprefix("feature")),\n    )\n    if not fields or set(dimension_columns) != {\n        f"{field}_dim_{index}" for field in fields for index in range(32)\n    }:\n        raise ValueError("Expected complete 32-dimensional vector fields")\n    summary = [name for name in train.columns if "_dim_" not in name]\n    x_train, c_train, x_valid, c_valid, state = prepare_features(\n        train.select(summary), valid.select(summary)\n    )\n    mappings, cardinalities = [], []\n    mapped_train = np.empty(c_train.shape, dtype=np.int64)\n    mapped_valid = np.empty(c_valid.shape, dtype=np.int64)\n    for index, cardinality in enumerate(state["cardinalities"]):\n        counts = np.bincount(c_train[:, index].numpy(), minlength=cardinality)\n        retained = np.flatnonzero(\n            (counts >= min_category_count) & (np.arange(cardinality) != 0)\n        )\n        mapping = np.zeros(cardinality, dtype=np.int64)\n        mapping[retained] = np.arange(1, len(retained) + 1)\n        mapped_train[:, index] = mapping[c_train[:, index].numpy()]\n        mapped_valid[:, index] = mapping[c_valid[:, index].numpy()]\n        mappings.append(mapping)\n        cardinalities.append(len(retained) + 1)\n    n_dense = x_train.shape[1]\n    x_train, missing_train = append_vectors(train, x_train, fields)\n    x_valid, missing_valid = append_vectors(valid, x_valid, fields)\n    state.update(\n        {\n            "original_cardinalities": state["cardinalities"],\n            "cardinalities": cardinalities,\n            "category_remappings": mappings,\n            "min_category_count": min_category_count,\n            "vector_fields": fields,\n            "n_dense": n_dense,\n            "missing_vectors_train": missing_train,\n            "missing_vectors_valid": missing_valid,\n            "validation_unknown_or_rare_category_rates": {\n                name: float((mapped_valid[:, index] == 0).mean())\n                for index, name in enumerate(state["categorical_columns"])\n            },\n        }\n    )\n    return (\n        x_train, torch.from_numpy(mapped_train),\n        x_valid, torch.from_numpy(mapped_valid), state,\n    )\n\n\ndef transform_custom_features(frame, state):\n    numeric, categorical = transform_features(frame, state)\n    remapped = np.empty(categorical.shape, dtype=np.int64)\n    for index, mapping in enumerate(state["category_remappings"]):\n        remapped[:, index] = mapping[categorical[:, index].numpy()]\n    numeric, _ = append_vectors(frame, numeric, state["vector_fields"])\n    return numeric, torch.from_numpy(remapped)\n\n\nclass RecommendationNetwork(nn.Module):\n    def __init__(\n        self, n_dense, cardinalities, n_vectors, attention=False, positive_rate=0.05\n    ):\n        super().__init__()\n        self.n_dense = n_dense\n        self.n_vectors = n_vectors\n        self.embeddings = nn.ModuleList(\n            nn.Embedding(cardinality, 8, padding_idx=0)\n            for cardinality in cardinalities\n        )\n        for embedding in self.embeddings:\n            nn.init.normal_(embedding.weight, std=0.02)\n            with torch.no_grad():\n                embedding.weight[0].zero_()\n        self.scalar_branch = nn.Sequential(\n            nn.Linear(n_dense, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2)\n        )\n        self.category_branch = nn.Sequential(\n            nn.Linear(8 * len(cardinalities), 64),\n            nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.3),\n        )\n        self.vector_weights = nn.Parameter(torch.empty(n_vectors, 33, 32))\n        for weight in self.vector_weights:\n            nn.init.xavier_uniform_(weight)\n        self.vector_bias = nn.Parameter(torch.zeros(n_vectors, 32))\n        self.missing_tokens = nn.Parameter(torch.randn(n_vectors, 32) * 0.02)\n        self.field_tokens = nn.Parameter(torch.randn(n_vectors, 32) * 0.02)\n        self.vector_transform = nn.Sequential(\n            nn.LayerNorm(32), nn.Linear(32, 64), nn.GELU(),\n            nn.Dropout(0.2), nn.Linear(64, 32),\n        )\n        self.vector_branch = nn.Sequential(\n            nn.Linear(n_vectors * 32, 64),\n            nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.2),\n        )\n        self.head = nn.Sequential(\n            nn.Linear(256, 64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64, 1)\n        )\n        with torch.no_grad():\n            self.head[-1].bias.fill_(float(np.log(positive_rate / (1 - positive_rate))))\n        self.attention = (\n            nn.MultiheadAttention(32, 4, dropout=0.2, batch_first=True)\n            if attention else None\n        )\n        self.attention_norm = nn.LayerNorm(32) if attention else None\n        self.attention_dropout = nn.Dropout(0.2)\n\n    def extract_features(self, numeric, categorical):\n        if self.training:\n            categorical = categorical.masked_fill(\n                torch.rand_like(categorical, dtype=torch.float32) < 0.3, 0\n            )\n        embedded = torch.cat(\n            [\n                layer(categorical[:, index])\n                for index, layer in enumerate(self.embeddings)\n            ],\n            dim=1,\n        )\n        vectors = numeric[:, self.n_dense :].reshape(-1, self.n_vectors, 34)\n        projected = F.gelu(\n            torch.einsum("bfi,fij->bfj", vectors[:, :, :33], self.vector_weights)\n            + self.vector_bias\n        )\n        tokens = torch.where(\n            vectors[:, :, 33:].bool(), projected, self.missing_tokens.unsqueeze(0)\n        ) + self.field_tokens\n        if self.attention is not None:\n            normalized = self.attention_norm(tokens)\n            interactions, _ = self.attention(\n                normalized, normalized, normalized, need_weights=False\n            )\n            tokens = tokens + self.attention_dropout(interactions)\n        tokens = tokens + self.vector_transform(tokens)\n        return torch.cat(\n            [\n                self.scalar_branch(numeric[:, : self.n_dense]),\n                self.category_branch(embedded),\n                self.vector_branch(tokens.flatten(1)),\n            ],\n            dim=1,\n        )\n    def forward(self, numeric, categorical):\n        return self.head(self.extract_features(numeric, categorical))\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--rows", type=int, default=150_000)\n    parser.add_argument("--data-dir", type=Path, default=ROOT / "data")\n    parser.add_argument("--sample-path", type=Path)\n    parser.add_argument("--epochs", type=int, default=30)\n    parser.add_argument("--patience", type=int, default=5)\n    parser.add_argument("--batch-size", type=int, default=1024)\n    parser.add_argument("--min-category-count", type=int, default=10)\n    parser.add_argument("--variant", choices=["both", "control", "attention"], default="both")\n    parser.add_argument("--device", choices=["auto", "cpu", "mps", "cuda"], default="auto")\n    parser.add_argument("--threads", type=int, default=4)\n    parser.add_argument("--output-dir", type=Path, default=ROOT / "artifacts/local-custom-v1")\n    args = parser.parse_args()\n    if min(\n        args.rows, args.epochs, args.patience, args.batch_size,\n        args.min_category_count, args.threads,\n    ) <= 0:\n        parser.error("All numerical arguments must be positive")\n    device = args.device\n    if device == "auto":\n        device = (\n            "cuda" if torch.cuda.is_available()\n            else "mps" if torch.backends.mps.is_available()\n            else "cpu"\n        )\n    if device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA requested but unavailable")\n    if device == "mps" and not torch.backends.mps.is_available():\n        raise RuntimeError("Apple MPS requested but unavailable")\n    torch.set_num_threads(args.threads)\n    started = time.monotonic()\n    sample_hash = None\n    if args.sample_path:\n        sample = pl.read_parquet(args.sample_path)\n        if sample.height != args.rows:\n            raise ValueError("Cached sample row count does not match --rows")\n        with args.sample_path.open("rb") as stream:\n            sample_hash = hashlib.file_digest(stream, "sha256").hexdigest()\n    else:\n        sample = load_sample(\n            args.data_dir / "train.parquet", args.rows, 42, expand_embeddings=True\n        )\n    sample = sample.sort("_time")\n    if sample.height < 2 or sample["_time"].null_count():\n        raise ValueError("Chronological validation requires nonmissing timestamps")\n    cutoff = sample["_time"][int(sample.height * 0.8)]\n    split = int(sample["_time"].search_sorted(cutoff, side="left"))\n    train = sample.slice(0, split)\n    valid = sample.slice(split)\n    for label, frame in [("training", train), ("validation", valid)]:\n        if set(frame[TARGET].unique().to_list()) != {0, 1}:\n            raise ValueError(f"{label} must contain both binary target classes")\n    x_train, c_train, x_valid, c_valid, state = prepare_custom_features(\n        train, valid, args.min_category_count\n    )\n    y_train = torch.tensor(train[TARGET].to_numpy().copy(), dtype=torch.float32)\n    y_valid = valid[TARGET].to_numpy().copy()\n    report = {\n        "seed": 42,\n        "sample_rows": sample.height,\n        "sample_sha256": sample_hash,\n        "training_rows": train.height,\n        "validation_rows": valid.height,\n        "cutoff": str(cutoff),\n        "input_features": sample.width - 2,\n        "dense_channels_after_preprocessing": state["n_dense"],\n        "vector_fields": state["vector_fields"],\n        "original_cardinalities": state["original_cardinalities"],\n        "retained_cardinalities": state["cardinalities"],\n        "min_category_count": args.min_category_count,\n        "validation_unknown_or_rare_category_rates": state["validation_unknown_or_rare_category_rates"],\n        "missing_vectors_train": state["missing_vectors_train"],\n        "missing_vectors_valid": state["missing_vectors_valid"],\n        "device": device,\n        "threads": args.threads,\n        "max_epochs": args.epochs,\n        "patience": args.patience,\n        "batch_size": args.batch_size,\n        "optimizer": {"name": "AdamW", "learning_rate": 0.0005, "weight_decay": 0.01},\n        "versions": {\n            name: version(name)\n            for name in ["torch", "numpy", "polars", "scikit-learn"]\n        },\n        "competition_submission": False,\n        "results": [],\n    }\n    positive_rate = float(y_train.mean().item())\n    del sample, train, valid\n    gc.collect()\n    output = args.output_dir\n    output.mkdir(parents=True, exist_ok=True)\n    joblib.dump(state, output / "preprocessing.joblib")\n    report["preparation_seconds"] = time.monotonic() - started\n    variants = ["control", "attention"] if args.variant == "both" else [args.variant]\n    for variant in variants:\n        torch.manual_seed(42)\n        model_config = {\n            "n_dense": state["n_dense"],\n            "cardinalities": state["cardinalities"],\n            "n_vectors": len(state["vector_fields"]),\n            "attention": variant == "attention",\n            "positive_rate": positive_rate,\n        }\n        model = RecommendationNetwork(**model_config).to(device)\n        optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)\n        variant_output = output / variant\n        variant_output.mkdir(exist_ok=True)\n        parameters = sum(parameter.numel() for parameter in model.parameters())\n        print(\n            f"Variant: {variant}; device: {device}; train: {len(y_train):,}; "\n            f"valid: {len(y_valid):,}; parameters: {parameters:,}; cutoff: {cutoff}",\n            flush=True,\n        )\n        fitted = fit_model(\n            model, model_config, optimizer, x_train, c_train, y_train,\n            x_valid, c_valid, y_valid, device=device, output=variant_output,\n            epochs=args.epochs, patience=args.patience, batch_size=args.batch_size,\n        )\n        result = {\n            "variant": variant,\n            "parameters": parameters,\n            "model_config": model_config,\n            **fitted,\n        }\n        report["results"].append(result)\n        (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n        print(\n            f"{variant}: AUC={fitted[\'validation_auc\']:.6f}; "\n            f"epoch={fitted[\'best_epoch\']}; training={fitted[\'training_seconds\']:.1f}s",\n            flush=True,\n        )\n        del model, optimizer\n        gc.collect()\n        if device == "mps":\n            torch.mps.empty_cache()\n        elif device == "cuda":\n            torch.cuda.empty_cache()\n    print("Requested validation results saved; no competition submission performed", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'feature_models.py': 'import math\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef pairwise_vector_features(vectors, pairs):\n    first, second = pairs\n    unit = vectors[:, :, :32]\n    cosine = (unit[:, first] * unit[:, second]).sum(dim=-1).clamp(-1, 1)\n    norms = vectors[:, :, 32].expm1()\n    present = vectors[:, first, 33] * vectors[:, second, 33]\n    dot = cosine * norms[:, first] * norms[:, second]\n    distance = (\n        norms[:, first].square() + norms[:, second].square() - 2 * dot\n    ).clamp_min(0).sqrt()\n    return torch.cat([\n        cosine * present,\n        dot.sign() * dot.abs().log1p().clamp_max(20) * present,\n        distance.log1p().clamp_max(20) * present,\n        present,\n    ], dim=1)\n\n\nclass FeatureEncoder(nn.Module):\n    def __init__(\n        self, n_numeric, n_dense, n_vectors, cardinalities, continuous_indices,\n        embedding_dimensions, numerical_encoding="none", bins=None,\n        vector_projection=False, vector_dropout=0.0, vector_crosses=False,\n        summary_groups=None, indicator_groups=None, numeric_dropout=0.0,\n    ):\n        super().__init__()\n        self.n_dense, self.n_vectors = n_dense, n_vectors\n        self.vector_dropout, self.vector_crosses = vector_dropout, vector_crosses\n        self.numeric_dropout = numeric_dropout\n        self.summary_groups = summary_groups or [[] for _ in range(n_vectors)]\n        self.indicator_groups = indicator_groups or [[] for _ in range(n_vectors)]\n        self.register_buffer("continuous", torch.tensor(continuous_indices, dtype=torch.int64))\n        remaining = [i for i in range(n_numeric) if i not in set(continuous_indices)]\n        self.register_buffer("remaining", torch.tensor(remaining, dtype=torch.int64))\n        self.embeddings = nn.ModuleList(\n            nn.Embedding(cardinality, dimension, padding_idx=0)\n            for cardinality, dimension in zip(cardinalities, embedding_dimensions)\n        )\n        for layer in self.embeddings:\n            nn.init.normal_(layer.weight, std=0.02)\n            with torch.no_grad():\n                layer.weight[0].zero_()\n        self.numerical = None\n        dimension = n_numeric + sum(embedding_dimensions)\n        if numerical_encoding != "none":\n            from rtdl_num_embeddings import PeriodicEmbeddings, PiecewiseLinearEmbeddings\n\n            if not continuous_indices:\n                raise ValueError("Numerical embeddings require continuous scalar features")\n            if numerical_encoding == "ple":\n                if bins is None or len(bins) != len(continuous_indices):\n                    raise ValueError("PLE requires one train-fitted bin array per feature")\n                self.numerical = PiecewiseLinearEmbeddings(\n                    [torch.tensor(values, dtype=torch.float32) for values in bins],\n                    8, activation=True, version="B",\n                )\n            elif numerical_encoding == "plr":\n                self.numerical = PeriodicEmbeddings(\n                    len(continuous_indices), 8, n_frequencies=8,\n                    frequency_init_scale=0.01, activation=True, lite=True,\n                )\n            else:\n                raise ValueError(f"Unknown numerical encoding: {numerical_encoding}")\n            dimension += len(continuous_indices) * 7\n        self.vector_weight = None\n        if vector_projection:\n            self.vector_weight = nn.Parameter(torch.empty(n_vectors, 32, 32))\n            for matrix in self.vector_weight:\n                nn.init.xavier_uniform_(matrix)\n            self.vector_bias = nn.Parameter(torch.zeros(n_vectors, 32))\n            self.vector_norm = nn.LayerNorm(32)\n        pairs = torch.triu_indices(n_vectors, n_vectors, offset=1)\n        self.register_buffer("pairs", pairs)\n        if vector_crosses:\n            dimension += pairs.shape[1] * 4\n        self.output_dim = dimension\n\n    def forward(self, numeric, categorical):\n        if self.training and self.numeric_dropout:\n            numeric = numeric.clone()\n            numeric[:, self.continuous] = F.dropout(\n                numeric[:, self.continuous], p=self.numeric_dropout, training=True\n            )\n        vectors = numeric[:, self.n_dense :].reshape(-1, self.n_vectors, 34)\n        if self.training and self.vector_dropout:\n            keep = torch.rand(\n                vectors.shape[:2], device=vectors.device\n            ) >= self.vector_dropout\n            numeric = numeric.clone()\n            vectors = vectors * keep.unsqueeze(-1)\n            for index in range(self.n_vectors):\n                if self.summary_groups[index]:\n                    numeric[:, self.summary_groups[index]] *= keep[:, index : index + 1]\n                if self.indicator_groups[index]:\n                    columns = self.indicator_groups[index]\n                    numeric[:, columns] = torch.where(\n                        keep[:, index : index + 1],\n                        numeric[:, columns], torch.ones_like(numeric[:, columns]),\n                    )\n        crossed = None\n        if self.vector_crosses:\n            crossed = pairwise_vector_features(vectors, self.pairs)\n        if self.vector_weight is not None:\n            directions = F.silu(self.vector_norm(\n                torch.einsum("bfi,fij->bfj", vectors[:, :, :32], self.vector_weight)\n                + self.vector_bias\n            )) * vectors[:, :, 33:]\n            vectors = torch.cat([directions, vectors[:, :, 32:]], dim=-1)\n        if self.vector_weight is not None or (self.training and self.vector_dropout):\n            numeric = torch.cat([numeric[:, : self.n_dense], vectors.flatten(1)], dim=1)\n        if self.numerical is not None:\n            numeric = torch.cat([\n                numeric[:, self.remaining],\n                self.numerical(numeric[:, self.continuous]).flatten(1),\n            ], dim=1)\n        if self.training:\n            categorical = categorical.masked_fill(\n                torch.rand_like(categorical, dtype=torch.float32) < 0.3, 0\n            )\n        parts = [\n            numeric,\n            *[layer(categorical[:, index]) for index, layer in enumerate(self.embeddings)],\n        ]\n        if crossed is not None:\n            parts.append(crossed)\n        return torch.cat(parts, dim=1)\n\n\nclass LowRankCross(nn.Module):\n    def __init__(self, dimension, rank=32, experts=1):\n        super().__init__()\n        self.experts = experts\n        self.down = nn.ModuleList(nn.Linear(dimension, rank, bias=False) for _ in range(experts))\n        self.up = nn.ModuleList(nn.Linear(rank, dimension, bias=False) for _ in range(experts))\n        self.bias = nn.Parameter(torch.zeros(experts, dimension))\n        if experts > 1:\n            self.middle = nn.ModuleList(nn.Linear(rank, rank, bias=False) for _ in range(experts))\n            self.gate = nn.Linear(dimension, experts, bias=False)\n\n    def forward(self, original, current):\n        if self.experts == 1:\n            return current + original * (self.up[0](self.down[0](current)) + self.bias[0])\n        weights = self.gate(current).softmax(dim=-1)\n        transformed = []\n        for index in range(self.experts):\n            low = self.down[index](current).tanh()\n            low = self.middle[index](low).tanh()\n            transformed.append(\n                original * (self.up[index](low) + self.bias[index])\n            )\n        return current + (\n            torch.stack(transformed, dim=1) * weights.unsqueeze(-1)\n        ).sum(dim=1)\n\n\nclass DCNv2Backbone(nn.Module):\n    def __init__(self, dimension, width=128, dropout=0.2, rank=32, experts=1):\n        super().__init__()\n        self.crosses = nn.ModuleList(LowRankCross(dimension, rank, experts) for _ in range(2))\n        self.deep = nn.Sequential(\n            nn.Linear(dimension, width), nn.ReLU(), nn.Dropout(dropout),\n            nn.Linear(width, width), nn.ReLU(), nn.Dropout(dropout),\n        )\n        self.output = nn.Linear(dimension + width, 1)\n\n    def forward(self, inputs):\n        crossed = inputs\n        for layer in self.crosses:\n            crossed = layer(inputs, crossed)\n        return self.output(torch.cat([crossed, self.deep(inputs)], dim=1))\n\n\nclass FinalMLPBackbone(nn.Module):\n    def __init__(self, dimension, width=128, dropout=0.2, heads=4, gating=True):\n        super().__init__()\n        if width % heads:\n            raise ValueError("FinalMLP width must be divisible by head count")\n        self.width, self.heads, self.gating = width, heads, gating\n        self.streams = nn.ModuleList(\n            nn.Sequential(\n                nn.Linear(dimension, width), nn.ReLU(), nn.Dropout(dropout),\n                nn.Linear(width, width), nn.ReLU(), nn.Dropout(dropout),\n            ) for _ in range(2)\n        )\n        if gating:\n            self.gates = nn.ModuleList(\n                nn.Sequential(\n                    nn.Linear(dimension, 64), nn.ReLU(),\n                    nn.Linear(64, dimension), nn.Sigmoid(),\n                ) for _ in range(2)\n            )\n        self.linear = nn.ModuleList(nn.Linear(width, 1, bias=False) for _ in range(2))\n        self.bilinear = nn.Parameter(torch.empty(heads, width // heads, width // heads))\n        for matrix in self.bilinear:\n            nn.init.xavier_uniform_(matrix)\n        self.bias = nn.Parameter(torch.zeros(1))\n\n    def forward(self, inputs):\n        streams = [\n            stream(inputs * (2 * self.gates[index](inputs) if self.gating else 1))\n            for index, stream in enumerate(self.streams)\n        ]\n        first, second = [stream.reshape(-1, self.heads, self.width // self.heads) for stream in streams]\n        interaction = torch.einsum("bhd,hde,bhe->b", first, self.bilinear, second).unsqueeze(1)\n        return self.linear[0](streams[0]) + self.linear[1](streams[1]) + interaction + self.bias\n\n\nclass HeterogeneousClassifier(nn.Module):\n    def __init__(\n        self, encoder_config, architecture="tabm", width=128, dropout=0.2,\n        members=8, positive_rate=0.05, rank=32, experts=1, gating=True,\n    ):\n        super().__init__()\n        self.encoder = FeatureEncoder(**encoder_config)\n        dimension = self.encoder.output_dim\n        if architecture == "tabm":\n            from tabm import TabM\n\n            self.backbone = TabM.make(\n                n_num_features=dimension, d_out=1, k=members,\n                n_blocks=2, d_block=width, dropout=dropout,\n            )\n            bias = self.backbone.output.bias\n        elif architecture == "dcnv2":\n            self.backbone = DCNv2Backbone(dimension, width, dropout, rank, experts)\n            bias = self.backbone.output.bias\n        elif architecture == "finalmlp":\n            self.backbone = FinalMLPBackbone(dimension, width, dropout, gating=gating)\n            bias = self.backbone.bias\n        else:\n            raise ValueError(f"Unknown architecture: {architecture}")\n        with torch.no_grad():\n            bias.fill_(float(np.log(positive_rate / (1 - positive_rate))))\n        self.architecture = architecture\n\n    def forward(self, numeric, categorical):\n        output = self.backbone(self.encoder(numeric, categorical))\n        return output.squeeze(-1) if self.architecture == "tabm" else output\n', 'tree_experiments.py': 'from datetime import datetime\nimport gc\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nfrom catboost import CatBoost, Pool\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport polars as pl\nimport pyarrow.parquet as pq\nfrom sklearn.decomposition import PCA\nfrom sklearn.metrics import log_loss, roc_auc_score\nimport torch\n\nfrom custom_network import pack_vectors\nfrom feature_models import pairwise_vector_features\nfrom train import TARGET, load_sample, model\n\n\ndef cross_frame(frame, fields):\n    pairs = torch.triu_indices(len(fields), len(fields), offset=1)\n    width = pairs.shape[1] * 4\n    values = np.empty((len(frame), width), dtype=np.float32)\n    for start in range(0, len(frame), 8192):\n        packed, _ = pack_vectors(frame.slice(start, 8192), fields)\n        values[start : start + len(packed)] = pairwise_vector_features(packed, pairs).numpy()\n    return pd.DataFrame(values, columns=[f"vector_cross_{i}" for i in range(width)])\n\n\ndef matrices(train, valid, fields, kind, output):\n    columns = [name for name in train.columns if name not in {TARGET, "_time", "_row_id"}]\n    categories = [name for name in columns if train.schema[name] == pl.String]\n    if kind in {"catboost-expanded", "lightgbm"}:\n        return train.select(columns).to_pandas(), valid.select(columns).to_pandas(), categories, []\n    summary = [name for name in columns if "_dim_" not in name]\n    x_train, x_valid = train.select(summary).to_pandas(), valid.select(summary).to_pandas()\n    embeddings = []\n    if kind == "catboost-crosses":\n        return (\n            pd.concat([x_train, cross_frame(train, fields)], axis=1),\n            pd.concat([x_valid, cross_frame(valid, fields)], axis=1),\n            categories, embeddings,\n        )\n    transforms = {}\n    additions_train, additions_valid = {}, {}\n    components = int(kind.rsplit("-", 1)[1]) if kind.startswith("pca-") else None\n    for field in fields:\n        dimensions = [f"{field}_dim_{i}" for i in range(32)]\n        raw_train = np.array(train.select(dimensions).to_numpy(), dtype=np.float32, order="C")\n        raw_valid = np.array(valid.select(dimensions).to_numpy(), dtype=np.float32, order="C")\n        present_train, present_valid = np.isfinite(raw_train).all(1), np.isfinite(raw_valid).all(1)\n        raw_train[~present_train], raw_valid[~present_valid] = 0, 0\n        additions_train[field + "_present"] = present_train.astype(np.float32)\n        additions_valid[field + "_present"] = present_valid.astype(np.float32)\n        if components is not None:\n            if present_train.sum() < components:\n                raise ValueError(f"{field} has insufficient training vectors for PCA-{components}")\n            transform = PCA(n_components=components, svd_solver="full")\n            transform.fit(raw_train[present_train])\n            encoded_train, encoded_valid = transform.transform(raw_train), transform.transform(raw_valid)\n            encoded_train[~present_train], encoded_valid[~present_valid] = 0, 0\n            for index in range(components):\n                name = f"{field}_pca_{index}"\n                additions_train[name], additions_valid[name] = encoded_train[:, index], encoded_valid[:, index]\n            transforms[field] = transform\n        else:\n            additions_train[field], additions_valid[field] = list(raw_train), list(raw_valid)\n            embeddings.append(field)\n    if transforms:\n        joblib.dump(transforms, output / "pca.joblib")\n    return (\n        pd.concat([x_train, pd.DataFrame(additions_train)], axis=1),\n        pd.concat([x_valid, pd.DataFrame(additions_valid)], axis=1),\n        categories, embeddings,\n    )\n\n\ndef run_trees(\n    path, output, cutoff, *, device="cuda", iterations=400,\n    light_iterations=1000, only=None, worker_path=None,\n    valid_end=None, fixed_budget=False,\n):\n    rows = pq.ParquetFile(path).metadata.num_rows\n    sample = load_sample(path, rows, 42, expand_embeddings=True).with_row_index("_row_id").sort("_time")\n    split = int(sample["_time"].search_sorted(datetime.fromisoformat(cutoff), side="left"))\n    if valid_end:\n        end = int(sample["_time"].search_sorted(datetime.fromisoformat(valid_end), side="left"))\n        sample = sample.slice(0, end)\n    train, valid = sample.slice(0, split), sample.slice(split)\n    fields = sorted({\n        name.rsplit("_dim_", 1)[0] for name in sample.columns if "_dim_" in name\n    }, key=lambda value: int(value.removeprefix("feature")))\n    y_train, y_valid = train[TARGET].to_numpy().copy(), valid[TARGET].to_numpy().copy()\n    if rows == 1537638 and cutoff == "2026-10-28T08:06:52.397000" and valid_end is None:\n        assert len(y_train) == 1230110 and len(y_valid) == 307528\n    report = {\n        "stage": "trees", "training_rows": len(y_train), "validation_rows": len(y_valid),\n        "cutoff": cutoff, "results": [], "competition_submission": False,\n    }\n    for kind in [\n        "catboost-expanded", "pca-8", "pca-16", "catboost-crosses",\n        "catboost-native-lda", "lightgbm",\n    ]:\n        if only is not None and kind not in only:\n            continue\n        destination = output / kind\n        destination.mkdir(parents=True, exist_ok=True)\n        started = time.monotonic()\n        x_train, x_valid, categories, embeddings = matrices(\n            train, valid, fields, kind, destination\n        )\n        preparation_seconds = time.monotonic() - started\n        input_features = x_train.shape[1]\n        print(f"START {kind}: {len(y_train)} train / {len(y_valid)} holdout", flush=True)\n        fit_started = time.monotonic()\n        if kind == "lightgbm":\n            with tempfile.TemporaryDirectory() as directory:\n                temporary = Path(directory)\n                x_train.to_parquet(temporary / "train.parquet", index=False)\n                x_valid.to_parquet(temporary / "valid.parquet", index=False)\n                np.savez(temporary / "labels.npz", train=y_train, valid=y_valid)\n                (temporary / "categories.json").write_text(json.dumps(categories))\n                del x_train, x_valid\n                gc.collect()\n                subprocess.run([\n                    sys.executable, str(worker_path or Path(__file__).with_name("lightgbm_worker.py")),\n                    "--input-dir", str(temporary), "--output-dir", str(destination),\n                    "--iterations", str(light_iterations),\n                    *(["--fixed-budget"] if fixed_budget else []),\n                ], check=True)\n            arrays = np.load(destination / "validation_predictions.npz")\n            probabilities = arrays["probabilities"]\n            worker = json.loads((destination / "worker-results.json").read_text())\n            best_iteration = worker["best_iteration"]\n            backend, library_version = worker["backend"], worker["library_version"]\n        else:\n            native = kind == "catboost-native-lda"\n            backend = "CPU" if native or device == "cpu" else "GPU"\n            if native:\n                estimator = CatBoost({\n                    "iterations": iterations, "depth": 6, "learning_rate": 0.06,\n                    "loss_function": "Logloss", "eval_metric": "AUC",\n                    "random_seed": 42, "thread_count": 4, "task_type": backend,\n                    "max_ctr_complexity": 1, "allow_writing_files": False, "verbose": 50,\n                    "embedding_processing": {"embedding_processing": {"default": ["LDA"]}},\n                })\n            else:\n                estimator = model(iterations, 4, backend)\n            training_pool = Pool(\n                x_train, label=y_train, cat_features=categories, embedding_features=embeddings\n            )\n            validation_pool = Pool(\n                x_valid, label=y_valid, cat_features=categories, embedding_features=embeddings\n            )\n            if fixed_budget:\n                estimator.fit(training_pool)\n            else:\n                estimator.fit(\n                    training_pool, eval_set=validation_pool, early_stopping_rounds=60\n                )\n            probabilities = estimator.predict(validation_pool, prediction_type="Probability")[:, 1]\n            estimator.save_model(str(destination / "model.cbm"))\n            best_iteration = estimator.tree_count_\n            import catboost\n\n            library_version = catboost.__version__\n            del training_pool, validation_pool\n        probabilities = np.asarray(probabilities)\n        if not np.isfinite(probabilities).all():\n            raise ValueError(f"Nonfinite {kind} predictions")\n        np.savez_compressed(\n            destination / "validation_predictions.npz",\n            y_true=y_valid, probabilities=probabilities,\n            row_ids=valid["_row_id"].to_numpy(),\n        )\n        result = {\n            "name": kind, "validation_auc": float(roc_auc_score(y_valid, probabilities)),\n            "validation_logloss": float(log_loss(y_valid, probabilities, labels=[0, 1])),\n            "preparation_seconds": preparation_seconds,\n            "training_and_prediction_seconds": time.monotonic() - fit_started,\n            "best_iteration": best_iteration, "backend": backend,\n            "library_version": library_version, "input_features": input_features,\n        }\n        report["results"].append(result)\n        (destination / "results.json").write_text(json.dumps(result, indent=2) + "\\n")\n        (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n        print(f"RESULT {kind}: AUC={result[\'validation_auc\']:.10f}", flush=True)\n        if kind != "lightgbm":\n            del estimator, x_train, x_valid\n        gc.collect()\n    report["status"] = "COMPLETE"\n    (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n', 'lightgbm_worker.py': 'import argparse\nimport json\nfrom pathlib import Path\nimport resource\nimport sys\n\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\n\n\nclass ArraySequence(lgb.Sequence):\n    batch_size = 8192\n\n    def __init__(self, path):\n        self.values = np.load(path, mmap_mode="r")\n\n    def __len__(self):\n        return len(self.values)\n\n    def __getitem__(self, index):\n        return np.asarray(self.values[index], dtype=np.float64)\n\n\ndef train_arrays(directory, output, iterations, threads, *, fixed_budget=False, full_training=False):\n    metadata = json.loads((directory / "metadata.json").read_text())\n    labels = np.load(directory / "labels.npz")\n    train_sequence = ArraySequence(directory / "train.npy")\n    valid_sequence = None if full_training else ArraySequence(directory / "valid.npy")\n    params = {\n        "objective": "binary", "metric": "auc", "learning_rate": 0.03,\n        "num_leaves": 31, "max_depth": -1, "min_data_in_leaf": 100,\n        "feature_fraction": 0.8, "lambda_l2": 1, "seed": 42,\n        "num_threads": threads, "verbosity": -1, "device_type": "cpu",\n        "force_col_wise": True, "histogram_pool_size": 512,\n    }\n    training = lgb.Dataset(\n        train_sequence, label=labels["train"], feature_name=metadata["columns"],\n        categorical_feature=metadata["categories"], params=params,\n    )\n    if fixed_budget:\n        booster = lgb.train(params, training, num_boost_round=iterations)\n    else:\n        validation = lgb.Dataset(\n            valid_sequence, label=labels["valid"], reference=training,\n            feature_name=metadata["columns"], categorical_feature=metadata["categories"],\n        )\n        booster = lgb.train(\n            params, training, num_boost_round=iterations,\n            valid_sets=[validation], valid_names=["validation"],\n            callbacks=[lgb.early_stopping(60), lgb.log_evaluation(50)],\n        )\n    selected_iteration = iterations if fixed_budget else booster.best_iteration\n    if not full_training:\n        probabilities = np.empty(len(valid_sequence), dtype=np.float64)\n        for start in range(0, len(valid_sequence), valid_sequence.batch_size):\n            end = min(start + valid_sequence.batch_size, len(valid_sequence))\n            probabilities[start:end] = booster.predict(\n                valid_sequence[start:end], num_iteration=selected_iteration,\n                num_threads=threads,\n            )\n        np.savez_compressed(\n            output / "validation_predictions.npz", y_true=labels["valid"],\n            probabilities=probabilities, row_ids=labels["row_ids"],\n        )\n    booster.save_model(str(output / "model.txt"))\n    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    (output / "worker-results.json").write_text(json.dumps({\n        "best_iteration": selected_iteration, "library_version": lgb.__version__,\n        "selection_mode": "fixed_budget" if fixed_budget else "validation",\n        "backend": "CPU isolated disk-backed process", "parameters": params,\n        "peak_rss_bytes": peak if sys.platform == "darwin" else peak * 1024,\n        "full_training": full_training, "training_rows": len(train_sequence),\n    }, indent=2) + "\\n")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    inputs = parser.add_mutually_exclusive_group(required=True)\n    inputs.add_argument("--input-dir", type=Path)\n    inputs.add_argument("--array-dir", type=Path)\n    inputs.add_argument("--predict-array", type=Path)\n    parser.add_argument("--model-file", type=Path)\n    parser.add_argument("--feature-columns", type=Path)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--iterations", type=int, default=1000)\n    parser.add_argument("--threads", type=int, default=2)\n    parser.add_argument("--fixed-budget", action="store_true")\n    parser.add_argument("--full-training", action="store_true")\n    args = parser.parse_args()\n    if args.full_training and (args.array_dir is None or not args.fixed_budget):\n        parser.error("--full-training requires --array-dir and --fixed-budget")\n    if args.predict_array is not None:\n        if args.model_file is None or args.feature_columns is None:\n            parser.error("--model-file and --feature-columns are required for prediction")\n        booster = lgb.Booster(model_file=str(args.model_file))\n        columns = json.loads(args.feature_columns.read_text())\n        matrix = np.load(args.predict_array, mmap_mode="r")\n        if matrix.ndim != 2 or matrix.shape[1] != booster.num_feature() or columns != booster.feature_name():\n            raise ValueError("Prediction features differ from the saved LightGBM model")\n        probabilities = np.empty(len(matrix), dtype=np.float64)\n        for start in range(0, len(matrix), 8192):\n            probabilities[start:start + 8192] = booster.predict(\n                np.asarray(matrix[start:start + 8192], dtype=np.float64),\n                num_threads=args.threads,\n            )\n        if not np.isfinite(probabilities).all() or not ((probabilities >= 0) & (probabilities <= 1)).all():\n            raise ValueError("Invalid saved-model LightGBM predictions")\n        np.save(args.output_dir / "probabilities.npy", probabilities)\n        return\n    if args.array_dir is not None:\n        train_arrays(\n            args.array_dir, args.output_dir, args.iterations, args.threads,\n            fixed_budget=args.fixed_budget,\n            full_training=args.full_training,\n        )\n        return\n    train = pd.read_parquet(args.input_dir / "train.parquet")\n    valid = pd.read_parquet(args.input_dir / "valid.parquet")\n    labels = np.load(args.input_dir / "labels.npz")\n    categories = json.loads((args.input_dir / "categories.json").read_text())\n    for column in categories:\n        dtype = pd.CategoricalDtype(categories=pd.unique(train[column]))\n        train[column] = train[column].astype(dtype)\n        valid[column] = valid[column].where(valid[column].isin(dtype.categories), None).astype(dtype)\n    estimator = lgb.LGBMClassifier(\n        objective="binary", metric="auc", n_estimators=args.iterations,\n        learning_rate=0.03, num_leaves=31, max_depth=-1,\n        min_child_samples=100, colsample_bytree=0.8,\n        reg_lambda=1, random_state=42, n_jobs=4, verbosity=-1, device_type="cpu",\n    )\n    if args.fixed_budget:\n        estimator.fit(train, labels["train"])\n    else:\n        estimator.fit(\n            train, labels["train"], eval_set=[(valid, labels["valid"])],\n            callbacks=[lgb.early_stopping(60), lgb.log_evaluation(50)],\n        )\n    probabilities = estimator.predict_proba(valid)[:, 1]\n    np.savez_compressed(\n        args.output_dir / "validation_predictions.npz",\n        y_true=labels["valid"], probabilities=probabilities,\n    )\n    estimator.booster_.save_model(str(args.output_dir / "model.txt"))\n    (args.output_dir / "worker-results.json").write_text(json.dumps({\n        "best_iteration": args.iterations if args.fixed_budget else estimator.best_iteration_,\n        "library_version": lgb.__version__,\n        "backend": "CPU isolated process",\n    }, indent=2) + "\\n")\n\n\nif __name__ == "__main__":\n    main()\n', 'local_tree_experiment.py': 'import argparse\nfrom datetime import datetime\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nimport numpy as np\nimport polars as pl\nimport pyarrow.parquet as pq\nfrom sklearn.metrics import log_loss, roc_auc_score\n\nfrom train import ROOT, TARGET, TIME, features\n\n\nCUTOFF = "2026-10-28T08:06:52.397000"\n\n\ndef prepare_arrays(\n    path, cache, output, reference_identity=None, *,\n    cutoff_string=CUTOFF, validation_end=None,\n):\n    cutoff = datetime.fromisoformat(cutoff_string)\n    parquet = pq.ParquetFile(path)\n    raw_identity = pl.read_parquet(path, columns=[TIME, TARGET, "request_id"]).with_row_index("_row_id")\n    identity = raw_identity.sort(TIME)\n    training = identity.filter(pl.col(TIME) < cutoff)\n    validation_mask = pl.col(TIME) >= cutoff\n    if validation_end is not None:\n        validation_mask &= pl.col(TIME) < datetime.fromisoformat(validation_end)\n    validation = identity.filter(validation_mask)\n    if not len(training) or not len(validation):\n        raise ValueError("Both full temporal partitions must be nonempty")\n    if reference_identity is not None:\n        expected = pl.read_parquet(reference_identity)\n        np.testing.assert_array_equal(\n            np.sort(validation["_row_id"].to_numpy()), np.sort(expected["_row_id"].to_numpy())\n        )\n        validation = raw_identity[expected["_row_id"].to_numpy()]\n        if not validation.select("_row_id", "request_id", TIME).equals(\n            expected.select("_row_id", "request_id", TIME)\n        ):\n            raise ValueError("Reference identities do not match the source data")\n    for frame in [training, validation]:\n        if set(frame[TARGET].unique().to_list()) != {0, 1}:\n            raise ValueError("Each temporal partition requires both target classes")\n    rows = parquet.metadata.num_rows\n    if rows == 1537638 and cutoff_string == CUTOFF and validation_end is None:\n        assert len(training) == 1230110 and len(validation) == 307528\n    training_ids, validation_ids = training["_row_id"].to_numpy(), validation["_row_id"].to_numpy()\n    validation.select("_row_id", "request_id", TIME).write_parquet(output / "validation-identities.parquet")\n    first = features(pl.from_arrow(next(parquet.iter_batches(batch_size=1))), expand_embeddings=True)\n    columns = first.columns\n    categories = [name for name, dtype in first.schema.items() if dtype == pl.String]\n    del first\n    category_frame = (\n        pl.scan_parquet(path).select(TIME, *categories)\n        .filter(pl.col(TIME) < cutoff).sort(TIME).collect()\n    )\n    vocabularies = {\n        name: category_frame[name].fill_null("__MISSING__").unique(maintain_order=True).to_list()\n        for name in categories\n    }\n    if any(len(values) >= 2**24 for values in vocabularies.values()):\n        raise ValueError("Categorical codes exceed exact float32 integer capacity")\n    del category_frame\n    mappings = {name: {value: i for i, value in enumerate(values)} for name, values in vocabularies.items()}\n    (output / "category-vocabularies.json").write_text(json.dumps(vocabularies) + "\\n")\n    del vocabularies\n    category_expressions = [\n        pl.col(name).replace_strict(mapping, default=-1, return_dtype=pl.Int32)\n        for name, mapping in mappings.items()\n    ]\n    del mappings\n    positions = {}\n    arrays = {}\n    for name, indices in [("train", training_ids), ("valid", validation_ids)]:\n        positions[name] = np.full(rows, -1, dtype=np.int64)\n        positions[name][indices] = np.arange(len(indices))\n        arrays[name] = np.lib.format.open_memmap(\n            cache / f"{name}.npy", mode="w+", dtype=np.float32,\n            shape=(len(indices), len(columns)),\n        )\n    written = {"train": 0, "valid": 0}\n    offset = 0\n    for batch in parquet.iter_batches(batch_size=8192):\n        frame = features(pl.from_arrow(batch), expand_embeddings=True)\n        frame = frame.with_columns(category_expressions)\n        matrix = frame.select(pl.all().cast(pl.Float32)).to_numpy()\n        for name in ["train", "valid"]:\n            destination = positions[name][offset : offset + len(frame)]\n            keep = destination >= 0\n            arrays[name][destination[keep]] = matrix[keep]\n            written[name] += int(keep.sum())\n        offset += len(frame)\n    if offset != rows or written != {"train": len(training), "valid": len(validation)}:\n        raise ValueError("Streaming preprocessing did not cover the complete dataset")\n    for array in arrays.values():\n        array.flush()\n    np.savez(\n        cache / "labels.npz", train=training[TARGET].to_numpy(),\n        valid=validation[TARGET].to_numpy(), row_ids=validation_ids,\n    )\n    (cache / "metadata.json").write_text(json.dumps({\n        "columns": columns, "categories": categories,\n    }) + "\\n")\n    protocol = {\n        "source_rows": rows, "training_rows": len(training), "validation_rows": len(validation),\n        "cutoff": cutoff_string, "validation_end": validation_end,\n        "input_features": len(columns),\n        "validation_labels_sha256": hashlib.sha256(\n            validation[TARGET].to_numpy().astype(np.uint8).tobytes()\n        ).hexdigest(),\n        "scope": "all rows in both full temporal partitions; no training subset",\n        "platform": platform.platform(), "competition_submission": False,\n    }\n    (output / "protocol.json").write_text(json.dumps(protocol, indent=2) + "\\n")\n    return protocol\n\n\ndef run(\n    path, output, *, iterations=1000, threads=2, reference_identity=None,\n    cutoff_string=CUTOFF, validation_end=None, fixed_budget=False,\n):\n    output.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    required = pq.ParquetFile(path).metadata.num_rows * 1127 * 4\n    if shutil.disk_usage(output).free < required + 2 * 1024**3:\n        raise OSError("Insufficient free disk space for the full disk-backed feature matrices")\n    with tempfile.TemporaryDirectory(prefix="streaming-lightgbm-", dir=output) as directory:\n        cache = Path(directory)\n        protocol = prepare_arrays(\n            path, cache, output, reference_identity,\n            cutoff_string=cutoff_string, validation_end=validation_end,\n        )\n        preparation_seconds = time.monotonic() - started\n        print(f"FULL DATA READY: {json.dumps(protocol)}", flush=True)\n        gc.collect()\n        destination = output / "lightgbm"\n        destination.mkdir(exist_ok=True)\n        fit_started = time.monotonic()\n        subprocess.run([\n            sys.executable, str(Path(__file__).with_name("lightgbm_worker.py")),\n            "--array-dir", str(cache), "--output-dir", str(destination),\n            "--iterations", str(iterations), "--threads", str(threads),\n            *(["--fixed-budget"] if fixed_budget else []),\n        ], check=True)\n        predictions = np.load(destination / "validation_predictions.npz")\n        probabilities = predictions["probabilities"]\n        if not np.isfinite(probabilities).all():\n            raise ValueError("Nonfinite full-data LightGBM predictions")\n        worker = json.loads((destination / "worker-results.json").read_text())\n        result = {\n            "name": "lightgbm", **worker,\n            "validation_auc": float(roc_auc_score(predictions["y_true"], probabilities)),\n            "validation_logloss": float(log_loss(predictions["y_true"], probabilities, labels=[0, 1])),\n            "training_and_prediction_seconds": time.monotonic() - fit_started,\n            "preparation_seconds": preparation_seconds,\n        }\n        (destination / "results.json").write_text(json.dumps(result, indent=2) + "\\n")\n        report = {\n            "stage": "local-trees", "status": "COMPLETE", "protocol": protocol,\n            "results": [result], "competition_submission": False,\n        }\n        (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n    print(json.dumps(result, indent=2), flush=True)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-dir", type=Path, default=ROOT / "data")\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--iterations", type=int, default=1000)\n    parser.add_argument("--threads", type=int, default=2)\n    parser.add_argument("--reference-identities", type=Path)\n    parser.add_argument("--cutoff", default=CUTOFF)\n    parser.add_argument("--validation-end")\n    parser.add_argument("--fixed-budget", action="store_true")\n    args = parser.parse_args()\n    if min(args.iterations, args.threads) <= 0:\n        parser.error("Iterations and CPU threads must be positive")\n    run(\n        args.data_dir / "train.parquet", args.output_dir,\n        iterations=args.iterations, threads=args.threads,\n        reference_identity=args.reference_identities,\n        cutoff_string=args.cutoff, validation_end=args.validation_end,\n        fixed_budget=args.fixed_budget,\n    )\n\n\nif __name__ == "__main__":\n    main()\n', 'full_experiments.py': 'import argparse\nimport gc\nimport hashlib\nfrom importlib.metadata import version\nimport json\nimport math\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nimport joblib\nimport numpy as np\nimport polars as pl\nimport pyarrow.parquet as pq\nimport torch\n\nfrom custom_network import prepare_custom_features\nfrom feature_models import HeterogeneousClassifier\nfrom tabm_experiment import TabMClassifier, fit_model_steps\nfrom train import ROOT, TARGET, load_sample\nfrom tree_experiments import run_trees\n\n\nPRIMARY_CUTOFF = "2026-10-28T08:06:52.397000"\nSECONDARY_CUTOFF = "2026-10-27T08:06:52.397000"\nSECONDARY_END = "2026-10-27T18:00:00"\n\n\ndef sha256(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef prepare_data(path, output, *, secondary=False, cutoff_override=None, end_override=None):\n    from datetime import datetime\n\n    started = time.monotonic()\n    total = pq.ParquetFile(path).metadata.num_rows\n    sample = (\n        load_sample(path, total, 42, expand_embeddings=True)\n        .with_row_index("_row_id").sort("_time")\n    )\n    if sample["_time"].null_count():\n        raise ValueError("Missing timestamps in experiment input")\n    cutoff = datetime.fromisoformat(\n        cutoff_override or (SECONDARY_CUTOFF if secondary else PRIMARY_CUTOFF)\n    )\n    split = int(sample["_time"].search_sorted(cutoff, side="left"))\n    if secondary or end_override:\n        end = int(sample["_time"].search_sorted(\n            datetime.fromisoformat(end_override or SECONDARY_END), side="left"\n        ))\n        sample = sample.slice(0, end)\n    train, valid = sample.slice(0, split), sample.slice(split)\n    training_row_ids = train["_row_id"].to_numpy().copy()\n    validation_row_ids = valid["_row_id"].to_numpy().copy()\n    train, valid = train.drop("_row_id"), valid.drop("_row_id")\n    if train.is_empty() or valid.is_empty():\n        raise ValueError("The fixed temporal cutoff does not split this dataset")\n    for name, frame in [("training", train), ("validation", valid)]:\n        if set(frame[TARGET].unique().to_list()) != {0, 1}:\n            raise ValueError(f"{name} requires both binary classes")\n    x_train, c_train, x_valid, c_valid, state = prepare_custom_features(train, valid, 10)\n    unique_counts = train.select(pl.col(state["numeric_columns"]).n_unique()).row(0)\n    continuous = [i for i, count in enumerate(unique_counts) if count > 16]\n    original_train, original_valid = [], []\n    category_counts = []\n    original_train_array = state["encoder"].transform(\n        train.select(state["categorical_columns"]).fill_null("__MISSING__").to_numpy()\n    ).astype(np.int64) + 1\n    original_valid_array = state["encoder"].transform(\n        valid.select(state["categorical_columns"]).fill_null("__MISSING__").to_numpy()\n    ).astype(np.int64) + 1\n    for index, cardinality in enumerate(state["original_cardinalities"]):\n        original_train.append(original_train_array[:, index].copy())\n        original_valid.append(original_valid_array[:, index].copy())\n        category_counts.append(np.bincount(original_train[-1], minlength=cardinality))\n    # Keep identifiers outside model inputs for paired/group-aware evaluation.\n    identity = (\n        pl.scan_parquet(path).select("request_id", "feature1")\n        .with_row_index("_row_id").collect()[validation_row_ids]\n    )\n    identity.write_parquet(output / "validation-identities.parquet")\n    protocol = {\n        "source_rows": total, "training_rows": train.height, "validation_rows": valid.height,\n        "cutoff": cutoff.isoformat(), "secondary": secondary,\n        "secondary_end": SECONDARY_END if secondary else None,\n        "validation_end": end_override or (SECONDARY_END if secondary else None),\n        "validation_labels_sha256": hashlib.sha256(\n            valid[TARGET].to_numpy().astype(np.uint8).tobytes()\n        ).hexdigest(),\n        "validation_identities_sha256": sha256(output / "validation-identities.parquet"),\n        "continuous_indices": continuous,\n        "versions": {\n            name: version(name)\n            for name in ["torch", "tabm", "rtdl_num_embeddings", "numpy", "polars", "scikit-learn"]\n        },\n        "preparation_seconds": time.monotonic() - started,\n        "competition_submission": False,\n    }\n    if not secondary and cutoff_override is None and total == 1537638:\n        assert train.height == 1230110 and valid.height == 307528\n    y_train = torch.tensor(train[TARGET].to_numpy().copy(), dtype=torch.float32)\n    y_valid = valid[TARGET].to_numpy().copy()\n    del sample, train, valid, identity\n    gc.collect()\n    joblib.dump(state, output / "preprocessing.joblib")\n    (output / "protocol.json").write_text(json.dumps(protocol, indent=2) + "\\n")\n    return {\n        "x_train": x_train, "c_train": c_train, "x_valid": x_valid, "c_valid": c_valid,\n        "y_train": y_train, "y_valid": y_valid, "state": state, "protocol": protocol,\n        "original_train": original_train, "original_valid": original_valid,\n        "category_counts": category_counts,\n        "training_row_ids": training_row_ids,\n        "validation_row_ids": validation_row_ids,\n    }\n\n\ndef category_view(data, minimum=10, drop_high=False, *, with_mappings=False):\n    state = data["state"]\n    if minimum == 10 and not drop_high:\n        result = (data["c_train"], data["c_valid"], state["cardinalities"])\n        return (*result, state["category_remappings"]) if with_mappings else result\n    train = np.empty(data["c_train"].shape, dtype=np.int64)\n    valid = np.empty(data["c_valid"].shape, dtype=np.int64)\n    cardinalities, mappings = [], []\n    for index, counts in enumerate(data["category_counts"]):\n        retained = np.flatnonzero((counts >= minimum) & (np.arange(len(counts)) != 0))\n        if drop_high and state["original_cardinalities"][index] > 1000:\n            retained = np.empty(0, dtype=np.int64)\n        mapping = np.zeros(len(counts), dtype=np.int64)\n        mapping[retained] = np.arange(1, len(retained) + 1)\n        train[:, index] = mapping[data["original_train"][index]]\n        valid[:, index] = mapping[data["original_valid"][index]]\n        cardinalities.append(len(retained) + 1)\n        mappings.append(mapping)\n    result = (torch.from_numpy(train), torch.from_numpy(valid), cardinalities)\n    return (*result, mappings) if with_mappings else result\n\n\ndef run_tabm(data, spec, output, *, device="cuda", epochs=30, interval=300, seed=42):\n    name = spec["name"]\n    destination = output / name\n    destination.mkdir(parents=True, exist_ok=True)\n    torch.manual_seed(seed)\n    train_categories, valid_categories, cardinalities, mappings = category_view(\n        data, spec.get("minimum", 10), spec.get("drop_high", False), with_mappings=True,\n    )\n    joblib.dump(mappings, destination / "category-remappings.joblib")\n    dims = [\n        min(32, math.ceil(cardinality ** 0.25)) if spec.get("adaptive") else spec.get("embedding_dim", 8)\n        for cardinality in cardinalities\n    ]\n    config = {\n        "n_numeric": data["x_train"].shape[1], "cardinalities": cardinalities,\n        "members": spec.get("members", 8), "width": spec.get("width", 128),\n        "blocks": 2, "dropout": spec.get("dropout", 0.2),\n        "category_dropout": 0.3, "positive_rate": float(data["y_train"].mean().item()),\n        "embedding_dimensions": dims,\n        "numeric_dropout": spec.get("numeric_dropout", 0.0),\n        "dropout_columns": data["protocol"]["continuous_indices"],\n    }\n    x_train, x_valid = data["x_train"], data["x_valid"]\n    numerical_encoding = spec.get("numerical_encoding", "none")\n    preprocessing_started = time.monotonic()\n    bins = None\n    if numerical_encoding == "ple":\n        from rtdl_num_embeddings import compute_bins\n\n        continuous = data["protocol"]["continuous_indices"]\n        bins = [\n            values.tolist() for values in compute_bins(\n                x_train[:, continuous], n_bins=16, verbose=False\n            )\n        ]\n    elif numerical_encoding == "quantile":\n        from sklearn.preprocessing import QuantileTransformer\n\n        continuous = data["protocol"]["continuous_indices"]\n        transformer = QuantileTransformer(\n            n_quantiles=min(256, len(x_train)), output_distribution="normal",\n            subsample=None, random_state=42, copy=False,\n        )\n        train_values = transformer.fit_transform(x_train[:, continuous].numpy())\n        valid_values = transformer.transform(x_valid[:, continuous].numpy())\n        x_train, x_valid = x_train.clone(), x_valid.clone()\n        x_train[:, continuous] = torch.from_numpy(train_values.astype(np.float32, copy=False))\n        x_valid[:, continuous] = torch.from_numpy(valid_values.astype(np.float32, copy=False))\n        joblib.dump(transformer, destination / "quantile-transformer.joblib")\n        del train_values, valid_values\n    adapted = (\n        numerical_encoding in {"ple", "plr"} or spec.get("vector_projection")\n        or spec.get("vector_dropout") or spec.get("vector_crosses")\n        or spec.get("architecture", "tabm") != "tabm"\n    )\n    if adapted:\n        state = data["state"]\n        summary_groups, indicator_groups = [], []\n        indicator_source = state["imputer"].indicator_.features_.tolist()\n        for field in state["vector_fields"]:\n            group = [\n                i for i, column in enumerate(state["numeric_columns"])\n                if column.startswith(field + "_")\n            ]\n            summary_groups.append(group)\n            indicator_groups.append([\n                len(state["numeric_columns"]) + position\n                for position, source in enumerate(indicator_source) if source in group\n            ])\n        encoder_config = {\n            "n_numeric": x_train.shape[1], "n_dense": state["n_dense"],\n            "n_vectors": len(state["vector_fields"]), "cardinalities": cardinalities,\n            "continuous_indices": data["protocol"]["continuous_indices"],\n            "embedding_dimensions": dims,\n            "numerical_encoding": numerical_encoding if numerical_encoding != "quantile" else "none",\n            "bins": bins, "vector_projection": bool(spec.get("vector_projection")),\n            "vector_dropout": spec.get("vector_dropout", 0.0),\n            "vector_crosses": bool(spec.get("vector_crosses")),\n            "summary_groups": summary_groups, "indicator_groups": indicator_groups,\n            "numeric_dropout": spec.get("numeric_dropout", 0.0),\n        }\n        config = {\n            "encoder_config": encoder_config, "architecture": spec.get("architecture", "tabm"),\n            "width": spec.get("width", 128), "dropout": spec.get("dropout", 0.2),\n            "members": spec.get("members", 8),\n            "positive_rate": float(data["y_train"].mean().item()),\n            "rank": spec.get("rank", 32), "experts": spec.get("experts", 1),\n            "gating": spec.get("gating", True),\n        }\n        model = HeterogeneousClassifier(**config).to(device)\n    else:\n        model = TabMClassifier(**config).to(device)\n    learning_rate = spec.get("learning_rate", 0.0005)\n    optimizer = torch.optim.AdamW(\n        model.parameters(), lr=learning_rate, weight_decay=spec.get("weight_decay", 0.01)\n    )\n    scheduler = None\n    if spec.get("cosine"):\n        period = 2 * math.ceil(len(data["y_train"]) / 1024)\n        scheduler = torch.optim.lr_scheduler.LambdaLR(\n            optimizer, lambda step: 0.1 + 0.45 * (1 + math.cos(math.pi * min(step / period, 1)))\n        )\n    variant_preparation_seconds = time.monotonic() - preprocessing_started\n    print(f"START {name}: {json.dumps(spec)}", flush=True)\n    metrics = fit_model_steps(\n        model, config, optimizer, x_train, train_categories, data["y_train"],\n        x_valid, valid_categories, data["y_valid"],\n        device=device, output=destination, epochs=epochs, patience=5, batch_size=1024,\n        validation_interval=interval, seed=seed, scheduler=scheduler,\n        fixed_budget=spec.get("fixed_budget", False),\n    )\n    saved_predictions = np.load(destination / "validation_predictions.npz")\n    np.savez_compressed(\n        destination / "validation_predictions.npz",\n        y_true=saved_predictions["y_true"],\n        probabilities=saved_predictions["probabilities"],\n        row_ids=data["validation_row_ids"],\n    )\n    result = {\n        "name": name, "spec": spec, "seed": seed,\n        "parameters": sum(parameter.numel() for parameter in model.parameters()),\n        "model_config": config, **metrics,\n        "variant_preparation_seconds": variant_preparation_seconds,\n    }\n    (destination / "results.json").write_text(json.dumps(result, indent=2) + "\\n")\n    print(f"RESULT {name}: AUC={result[\'validation_auc\']:.10f}", flush=True)\n    del model, optimizer, train_categories, valid_categories, x_train, x_valid\n    gc.collect()\n    if device == "cuda":\n        torch.cuda.empty_cache()\n    return result\n\n\ndef optimization_specs():\n    return [\n        {"name": "epoch-checkpoint-control", "epoch_only": True},\n        {"name": "reference"},\n        {"name": "learning-rate-1e4", "learning_rate": 0.0001},\n        {"name": "dropout-035", "dropout": 0.35},\n        {"name": "weight-decay-003", "weight_decay": 0.03},\n        {"name": "width-64", "width": 64},\n        {"name": "numeric-dropout-01", "numeric_dropout": 0.1},\n        {"name": "cosine-two-epochs", "cosine": True},\n        {"name": "minimum-category-50", "minimum": 50},\n        {"name": "embedding-dim-4", "embedding_dim": 4},\n        {"name": "adaptive-category-dims", "adaptive": True},\n        {"name": "without-high-cardinality", "drop_high": True},\n    ]\n\n\ndef feature_specs():\n    return [\n        {"name": "reference"},\n        {"name": "ple-16bins-d8", "numerical_encoding": "ple"},\n        {"name": "plr-d8", "numerical_encoding": "plr"},\n        {"name": "quantile-normal", "numerical_encoding": "quantile"},\n        {"name": "vector-projection-ln", "vector_projection": True},\n        {"name": "vector-dropout-01", "vector_dropout": 0.1},\n        {"name": "vector-crosses", "vector_crosses": True},\n    ]\n\n\ndef architecture_specs(base):\n    return [\n        {**base, "name": "ensemble-16", "members": 16},\n        {**base, "name": "ensemble-32", "members": 32},\n        {**base, "name": "dcnv2-r32", "architecture": "dcnv2", "rank": 32},\n        {**base, "name": "dcnv2-r64", "architecture": "dcnv2", "rank": 64},\n        {**base, "name": "dcnv2-moe4", "architecture": "dcnv2", "rank": 32, "experts": 4},\n        {**base, "name": "finalmlp-gated", "architecture": "finalmlp", "gating": True},\n        {**base, "name": "finalmlp-ungated", "architecture": "finalmlp", "gating": False},\n        {\n            **base, "name": "finalmlp-alternate-lr", "architecture": "finalmlp",\n            "gating": True, "learning_rate": 0.0001 if base.get("learning_rate", 0.0005) != 0.0001 else 0.0005,\n        },\n    ]\n\n\ndef run_confirmation(path, output, specs, *, device="cuda", epochs=30, interval=300, fold=None):\n    report = {"stage": "confirmation", "folds": [], "results": [], "competition_submission": False}\n    if fold is None:\n        spec_path = output / "confirmation-specs.json"\n        spec_path.write_text(json.dumps(specs, indent=2) + "\\n")\n        worker = globals().get("FROZEN_EXPERIMENT_FILE", Path(__file__).resolve())\n        report["summary"] = []\n        for selected in ["primary", "earlier"]:\n            subprocess.run([\n                sys.executable, str(worker), "--output-dir", str(output),\n                "--device", device, "--stage", "confirmation",\n                "--spec-file", str(spec_path), "--confirmation-fold", selected,\n                "--confirmation-input-file", str(path),\n                "--epochs", str(epochs), "--validation-interval", str(interval),\n            ], check=True)\n            partial = json.loads((output / f"{selected}-results.json").read_text())\n            if partial.get("status") != "COMPLETE":\n                raise ValueError(f"Incomplete confirmation fold: {selected}")\n            for key in ["folds", "results", "summary"]:\n                report[key].extend(partial[key])\n            (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n        report["status"] = "COMPLETE"\n        (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n        return report\n    if fold not in {"primary", "earlier"}:\n        raise ValueError(f"Unknown confirmation fold: {fold}")\n    report_path = output / f"{fold}-results.json"\n    for secondary in [fold == "earlier"]:\n        fold = "earlier" if secondary else "primary"\n        destination = output / fold\n        destination.mkdir(parents=True, exist_ok=True)\n        data = prepare_data(path, destination, secondary=secondary)\n        report["folds"].append({"name": fold, **data["protocol"]})\n        for recipe in specs:\n            for seed in [42, 43, 44]:\n                spec = {**recipe, "name": f"{recipe[\'name\']}-seed-{seed}"}\n                result = run_tabm(\n                    data, spec, destination, device=device, epochs=epochs, interval=interval, seed=seed,\n                )\n                result["fold"] = fold\n                result["recipe"] = recipe["name"]\n                report["results"].append(result)\n                report_path.write_text(json.dumps(report, indent=2) + "\\n")\n        del data\n        gc.collect()\n    report["summary"] = []\n    for fold in [entry["name"] for entry in report["folds"]]:\n        for recipe in specs:\n            values = [\n                row["validation_auc"] for row in report["results"]\n                if row["fold"] == fold and row["recipe"] == recipe["name"]\n            ]\n            report["summary"].append({\n                "fold": fold, "recipe": recipe["name"], "mean_auc": float(np.mean(values)),\n                "std_auc": float(np.std(values, ddof=1)), "seed_aucs": values,\n            })\n    report["status"] = "COMPLETE"\n    report_path.write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n\n\ndef fit_stacking_base(path, output, plan, cutoff, end, base, *, device="cuda"):\n    output.mkdir(parents=True, exist_ok=True)\n    if base == "neural":\n        data = prepare_data(path, output, cutoff_override=cutoff, end_override=end)\n        recipe = {\n            **plan["neural_recipe"], "name": "neural-fixed-budget", "fixed_budget": True,\n        }\n        return run_tabm(\n            data, recipe, output, device=device,\n            epochs=max(1, int(math.ceil(plan["fixed_neural_epochs"]))),\n            interval=300, seed=42,\n        )\n    if base != "tree":\n        raise ValueError(f"Unknown stacking base: {base}")\n    tree_output = output / "tree"\n    tree_output.mkdir(exist_ok=True)\n    if plan["tree_kind"] == "lightgbm":\n        from local_tree_experiment import run as run_local_tree\n\n        identity = output / "validation-identities.parquet"\n        return run_local_tree(\n            path, tree_output, iterations=plan["tree_iterations"],\n            reference_identity=identity if identity.is_file() else None,\n            cutoff_string=cutoff, validation_end=end, fixed_budget=True,\n        )\n    return run_trees(\n        path, tree_output, cutoff, device=device,\n        iterations=plan["tree_iterations"], light_iterations=plan["tree_iterations"],\n        only=[plan["tree_kind"]], valid_end=end, fixed_budget=True,\n    )\n\n\ndef finish_stacking(output, report):\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.metrics import log_loss, roc_auc_score\n\n    features, labels, identities, outer = [], [], [], None\n    expected_periods = [\n        ("2026-10-26T12:00:00", "2026-10-27T00:00:00"),\n        ("2026-10-27T00:00:00", "2026-10-27T12:00:00"),\n        ("2026-10-27T12:00:00", PRIMARY_CUTOFF),\n        (PRIMARY_CUTOFF, None),\n    ]\n    if [(fold["cutoff"], fold["end"]) for fold in report["folds"]] != expected_periods:\n        raise ValueError("Stacking requires all three forward OOF blocks and the outer holdout")\n    for index, fold in enumerate(report["folds"]):\n        destination = output / ("outer" if fold["end"] is None else f"oof-{index + 1}")\n        with (\n            np.load(destination / fold["neural"]["name"] / "validation_predictions.npz") as neural,\n            np.load(destination / "tree" / fold["tree"]["name"] / "validation_predictions.npz") as tree,\n        ):\n            np.testing.assert_array_equal(neural["y_true"], tree["y_true"])\n            np.testing.assert_array_equal(neural["row_ids"], tree["row_ids"])\n            target, row_ids = neural["y_true"].copy(), neural["row_ids"].copy()\n            probability_matrix = np.column_stack([\n                neural["probabilities"], tree["probabilities"],\n            ])\n        clipped = np.clip(probability_matrix, 1e-6, 1 - 1e-6)\n        logits = np.log(clipped / (1 - clipped))\n        if fold["end"] is None:\n            outer = (logits, target, row_ids, probability_matrix)\n        else:\n            features.append(logits)\n            labels.append(target)\n            identities.append(row_ids)\n    combined_ids = np.concatenate(identities)\n    if len(np.unique(combined_ids)) != len(combined_ids):\n        raise ValueError("OOF prediction blocks overlap")\n    if len(np.unique(outer[2])) != len(outer[2]) or np.intersect1d(combined_ids, outer[2]).size:\n        raise ValueError("Outer holdout identities duplicate or overlap OOF rows")\n    meta = LogisticRegression(C=1.0, max_iter=1000, random_state=42)\n    meta.fit(np.concatenate(features), np.concatenate(labels))\n    joblib.dump(meta, output / "meta-model.joblib")\n    probabilities = meta.predict_proba(outer[0])[:, 1]\n    np.savez_compressed(\n        output / "validation_predictions.npz", y_true=outer[1],\n        probabilities=probabilities, row_ids=outer[2],\n    )\n    np.savez_compressed(\n        output / "oof_predictions.npz", logits=np.concatenate(features),\n        y_true=np.concatenate(labels), row_ids=combined_ids,\n    )\n    report.update({\n        "stage": "stacking", "status": "COMPLETE", "oof_rows": len(combined_ids),\n        "validation_rows": len(outer[1]),\n        "validation_auc": float(roc_auc_score(outer[1], probabilities)),\n        "validation_logloss": float(log_loss(outer[1], probabilities, labels=[0, 1])),\n        "matched_neural_auc": float(roc_auc_score(outer[1], outer[3][:, 0])),\n        "matched_tree_auc": float(roc_auc_score(outer[1], outer[3][:, 1])),\n        "matched_equal_blend_auc": float(roc_auc_score(outer[1], outer[3].mean(axis=1))),\n        "meta_coefficients": meta.coef_.tolist(), "meta_intercept": meta.intercept_.tolist(),\n    })\n    (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n\n\ndef run_stacking(path, output, plan, *, device="cuda", outer_only=False):\n    periods = [\n        ("2026-10-26T12:00:00", "2026-10-27T00:00:00"),\n        ("2026-10-27T00:00:00", "2026-10-27T12:00:00"),\n        ("2026-10-27T12:00:00", PRIMARY_CUTOFF),\n        (PRIMARY_CUTOFF, None),\n    ]\n    fixed_epochs = max(1, int(math.ceil(plan["fixed_neural_epochs"])))\n    report = {\n        "stage": "stacking-base-fits" if outer_only else "stacking",\n        "folds": [], "competition_submission": False, "fixed_neural_epochs": fixed_epochs,\n    }\n    plan_path = output / "stacking-plan.json"\n    plan_path.write_text(json.dumps(plan, indent=2) + "\\n")\n    worker = globals().get("FROZEN_EXPERIMENT_FILE", Path(__file__).resolve())\n    for index, (cutoff, end) in enumerate(periods):\n        if outer_only and end is not None:\n            continue\n        destination = output / ("outer" if end is None else f"oof-{index + 1}")\n        destination.mkdir(parents=True, exist_ok=True)\n        for base in ["neural", "tree"]:\n            subprocess.run([\n                sys.executable, str(worker), "--data-dir", str(path.parent),\n                "--output-dir", str(destination), "--device", device,\n                "--stage", "stacking", "--spec-file", str(plan_path),\n                "--stacking-base", base, "--stacking-input-file", str(path),\n                "--cutoff", cutoff,\n                *(["--validation-end", end] if end else []),\n            ], check=True)\n        protocol = json.loads((destination / "protocol.json").read_text())\n        neural = json.loads((destination / "neural-fixed-budget/results.json").read_text())\n        tree_report = json.loads((destination / "tree/results.json").read_text())\n        report["folds"].append({\n            "cutoff": cutoff, "end": end, "protocol": protocol,\n            "neural": neural, "tree": tree_report["results"][0],\n        })\n        (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n    if outer_only:\n        report["status"] = "COMPLETE"\n        (output / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n        return report\n    return finish_stacking(output, report)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-dir", type=Path, default=ROOT / "data")\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--device", choices=["cpu", "cuda"], default="cuda")\n    parser.add_argument(\n        "--stage", choices=["optimization", "features", "trees", "recipes", "combined", "confirmation", "stacking"],\n        default="optimization",\n    )\n    parser.add_argument("--spec-file", type=Path)\n    parser.add_argument("--secondary", action="store_true")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--only", action="append")\n    parser.add_argument("--epochs", type=int, default=30)\n    parser.add_argument("--validation-interval", type=int, default=300)\n    parser.add_argument("--stacking-base", choices=["neural", "tree"])\n    parser.add_argument("--stacking-input-file", type=Path)\n    parser.add_argument("--confirmation-fold", choices=["primary", "earlier"])\n    parser.add_argument("--confirmation-input-file", type=Path)\n    parser.add_argument("--cutoff")\n    parser.add_argument("--validation-end")\n    parser.add_argument("--outer-only", action="store_true")\n    args = parser.parse_args()\n    if args.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("The full Kaggle experiment requires CUDA")\n    torch.set_num_threads(4)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    if args.stage == "trees":\n        run_trees(\n            args.data_dir / "train.parquet", args.output_dir, PRIMARY_CUTOFF,\n            device=args.device, only=args.only,\n            worker_path=globals().get("FROZEN_LIGHTGBM_WORKER_FILE"),\n        )\n        return\n    if args.stage in {"confirmation", "stacking"}:\n        if args.spec_file is None:\n            parser.error("--spec-file is required for confirmation/stacking")\n        plan = json.loads(args.spec_file.read_text())\n        if args.stage == "confirmation":\n            run_confirmation(\n                args.confirmation_input_file or args.data_dir / "train.parquet",\n                args.output_dir, plan,\n                device=args.device, epochs=args.epochs, interval=args.validation_interval,\n                fold=args.confirmation_fold,\n            )\n        else:\n            if args.stacking_base:\n                if args.cutoff is None:\n                    parser.error("--cutoff is required with --stacking-base")\n                fit_stacking_base(\n                    args.stacking_input_file or args.data_dir / "train.parquet",\n                    args.output_dir, plan,\n                    args.cutoff, args.validation_end, args.stacking_base, device=args.device,\n                )\n            else:\n                run_stacking(\n                    args.data_dir / "train.parquet", args.output_dir, plan,\n                    device=args.device, outer_only=args.outer_only,\n                )\n        return\n    data = prepare_data(\n        args.data_dir / "train.parquet", args.output_dir, secondary=args.secondary\n    )\n    report = {"stage": args.stage, "protocol": data["protocol"], "results": [], "competition_submission": False}\n    if args.stage in {"recipes", "combined"}:\n        if args.spec_file is None:\n            parser.error("--spec-file is required for recipes")\n        specs = json.loads(args.spec_file.read_text())\n    else:\n        specs = optimization_specs() if args.stage == "optimization" else feature_specs()\n    if args.only:\n        selected = set(args.only)\n        names = {spec["name"] for spec in specs}\n        if not selected <= names:\n            raise ValueError(f"Unknown experiment names: {selected - names}")\n        specs = [spec for spec in specs if spec["name"] in selected]\n    for spec in specs:\n        result = run_tabm(\n            data, spec, args.output_dir, device=args.device,\n            epochs=args.epochs,\n            interval=math.ceil(len(data["y_train"]) / 1024) if spec.get("epoch_only") else args.validation_interval,\n            seed=spec.get("seed", args.seed),\n        )\n        report["results"].append(result)\n        (args.output_dir / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n    if args.stage == "combined":\n        best = max(report["results"], key=lambda entry: entry["validation_auc"])\n        report["selected_combination"] = {"name": best["name"], "spec": best["spec"]}\n        for spec in architecture_specs(best["spec"]):\n            result = run_tabm(\n                data, spec, args.output_dir, device=args.device,\n                epochs=args.epochs, interval=args.validation_interval,\n            )\n            report["results"].append(result)\n            (args.output_dir / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n    report["status"] = "COMPLETE"\n    (args.output_dir / "results.json").write_text(json.dumps(report, indent=2) + "\\n")\n\n\nif __name__ == "__main__":\n    main()\n', 'request_features.py': '"""Label-free within-request context features appended to raw ShareChat parquet files.\n\nEvery row of a request shares one timestamp and requests never cross the\ntrain/holdout/test boundaries, so these features are computed per request and\nuse no labels. request_id itself is never a model input.\n"""\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport polars as pl\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\n\nITEM = "feature18"\nRANKED = ["feature418", "feature419", "feature411", "feature410", "feature430",\n          "feature431", "feature417", "feature352", "feature346", "feature480"]\n\n\ndef request_expressions():\n    group = "request_id"\n    size = pl.len().over(group)\n    exprs = [size.alias("rq_size"),\n             pl.col(ITEM).n_unique().over(group).alias("rq_items"),\n             pl.len().over([group, ITEM]).alias("rq_item_dup")]\n    for c in RANKED:\n        exprs += [\n            (pl.col(c).rank("average").over(group) / size).alias(f"rq_rank_{c}"),\n            (pl.col(c) - pl.col(c).mean().over(group)).alias(f"rq_dev_{c}"),\n            (pl.col(c) - pl.col(c).max().over(group)).alias(f"rq_gap_{c}"),\n        ]\n    return [e.cast(pl.Float32) for e in exprs]\n\n\ndef request_features(frame):\n    """Return request features in the same row order as ``frame``."""\n    return frame.select(request_expressions())\n\n\ndef augment(source, destination, batch_size=65536):\n    """Copy ``source`` to ``destination`` with request features appended, preserving row order."""\n    narrow = pl.read_parquet(source, columns=["request_id", ITEM, *RANKED])\n    extra = request_features(narrow)\n    del narrow\n    parquet = pq.ParquetFile(source)\n    if parquet.metadata.num_rows != len(extra):\n        raise ValueError("Request features do not cover every source row")\n    overlap = set(extra.columns) & set(parquet.schema_arrow.names)\n    if overlap:\n        raise ValueError(f"Source already contains request features: {sorted(overlap)}")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    writer, offset = None, 0\n    try:\n        for batch in parquet.iter_batches(batch_size=batch_size):\n            table = pa.Table.from_batches([batch])\n            appended = extra.slice(offset, len(table)).to_arrow()\n            for name in appended.column_names:\n                table = table.append_column(name, appended[name])\n            if writer is None:\n                writer = pq.ParquetWriter(destination, table.schema)\n            writer.write_table(table)\n            offset += len(table)\n    finally:\n        if writer is not None:\n            writer.close()\n    if offset != len(extra):\n        raise ValueError("Augmented parquet does not cover every source row")\n    return extra.columns\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-dir", type=Path, required=True)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    args = parser.parse_args()\n    for name in ["train", "test"]:\n        columns = augment(args.data_dir / f"{name}.parquet", args.output_dir / f"{name}.parquet")\n        print(f"{name}: appended {len(columns)} request features", flush=True)\n    sample = args.data_dir / "sample_submission.csv"\n    target = args.output_dir / "sample_submission.csv"\n    if sample.is_file() and not target.exists():\n        target.symlink_to(sample.resolve())\n\n\nif __name__ == "__main__":\n    main()\n'}
frozen = Path("/kaggle/working/frozen-source")
frozen.mkdir()
for name, source in sources.items():
    (frozen / name).write_text(source)
specs = [{'name': 'cross-masked-plr', 'vector_crosses': True, 'drop_high': True, 'numerical_encoding': 'plr'}, {'name': 'ensemble-32', 'vector_crosses': True, 'drop_high': True, 'numerical_encoding': 'plr', 'members': 32}, {'name': 'dcnv2-r32', 'vector_crosses': True, 'drop_high': True, 'numerical_encoding': 'plr', 'architecture': 'dcnv2', 'rank': 32}]
spec_path = frozen / "specs.json"
spec_path.write_text(json.dumps(specs, indent=2) + "\n")
data = [p.parent for p in Path("/kaggle/input").rglob("train.parquet") if (p.parent / "sample_submission.csv").is_file()]
if len(data) != 1:
    raise RuntimeError(f"Expected competition data: {data}")
augmented = Path("/kaggle/working/request-augmented")
augmented.mkdir()
subprocess.run([
    sys.executable, "-c",
    "import sys; from pathlib import Path; sys.path.insert(0, sys.argv[1]); "
    "from request_features import augment; "
    "print(augment(Path(sys.argv[2]), Path(sys.argv[3]))[:3], flush=True)",
    str(frozen), str(data[0] / "train.parquet"), str(augmented / "train.parquet"),
], check=True)
subprocess.run("free -m; df -h /tmp /kaggle/working", shell=True, check=False)
try:
    subprocess.run([
        sys.executable, str(frozen / "full_experiments.py"),
        "--data-dir", str(augmented), "--stage", "recipes", "--device", "cuda",
        "--output-dir", "/kaggle/working/t024", "--spec-file", str(spec_path),
    ], check=True)
finally:
    shutil.rmtree(augmented, ignore_errors=True)
print(Path("/kaggle/working/t024/results.json").read_text(), flush=True)
